# Brain-Act Full Analysis Pipeline — v2
### Brain States · Lempel-Ziv Complexity · Perturbational Complexity Index
**Domains:** Neural Firing Rates (AdEx/MF) and BOLD fMRI  
**Conditions:** Control (CNT) · Minimally Conscious State (MCS) · Unresponsive Wakefulness Syndrome (UWS)  

---

## What this notebook does

This notebook is a **self-contained analysis course** for the Brain-Act simulation dataset. It covers three complementary complexity measures used to probe disorders of consciousness (DoC) in whole-brain computational models:

1. **Phase-coherence Brain States** (firing rates *and* BOLD): How does the brain's instantaneous spatial synchrony pattern evolve over time, and how do these patterns differ between CNT, MCS, and UWS?

2. **Lempel-Ziv Complexity (LZc)** (firing rates *and* BOLD): How algorithmically complex is the brain's spatiotemporal activity? Higher complexity → more distinct patterns → broader repertoire of brain states.

3. **Perturbational Complexity Index (PCI)** (firing rates *only*): What is the richness of the brain's *causal* response to a TMS-like perturbation? This measure, introduced by Casali et al. (2013), is the gold standard for consciousness detection independent of sensory processing.

The notebook is structured so that each measure is **first explained mathematically and intuitively**, then computed, then visualised.

> **Data source.** The notebook tries to load Brain-Act cached simulation outputs first. If not found it generates synthetic data (with realistic statistical properties) so the full analysis pipeline can be followed without requiring a simulation run.

### References
- Casali et al. (2013). *A theoretically based index of consciousness independent of sensory processing and behavior.* Science Translational Medicine.
- Lempel & Ziv (1976). *On the complexity of finite sequences.* IEEE Trans. Inf. Theory.
- Kusch et al. (2023). Brain-Act: A whole-brain activity and connectivity tool.

---
## 0. Setup and Imports

We import TVBToolkit's analysis and complexity modules, which contain all the algorithms described in this notebook.

In [ ]:
from __future__ import annotations

import os
import sys
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# ── Suppress noisy third-party warnings ──────────────────────────────────────
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# ── Resolve TVBToolkit source root ────────────────────────────────────────────
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists() and (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib-cache')
os.environ.setdefault('TVB_USER_HOME', str(PROJECT_ROOT / '.tvb-temp'))

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# ── TVBToolkit analysis modules ───────────────────────────────────────────────
from tvbtoolkit.analysis.brain_states import (
    summarize_brain_states,
    centers_to_matrices,
    sfc_sort_centroids,
    BrainStateSummary,
)
from tvbtoolkit.analysis.spectral import (
    psd_per_region,
    phase_coherence_validity,
)
from tvbtoolkit.complexity.measures import (
    lzc_multichannel,
    pci_casali_like_multi_trial,
)

print('TVBToolkit imported successfully.')
print(f'Project root: {PROJECT_ROOT}')

import matplotlib
try:
    matplotlib.font_manager.findSystemFonts()  # ensure cache built
    _fonts = [f.name for f in matplotlib.font_manager.fontManager.ttflist]
    _use_font = 'Helvetica' if 'Helvetica' in _fonts else 'Arial' if 'Arial' in _fonts else 'DejaVu Sans'
except Exception:
    _use_font = 'DejaVu Sans'

plt.rcParams.update({
    # ── Nature Neuroscience typography ───────────────────────────────
    'font.family':         'sans-serif',
    'font.sans-serif':     ['Helvetica', 'Arial', 'DejaVu Sans'],
    'font.size':            7,
    'axes.labelsize':       8,
    'axes.titlesize':       8,
    'xtick.labelsize':      7,
    'ytick.labelsize':      7,
    'legend.fontsize':      7,
    # ── Clean axes ───────────────────────────────────────────────────
    'axes.linewidth':       0.8,
    'axes.spines.top':      False,
    'axes.spines.right':    False,
    'xtick.major.width':    0.8,
    'ytick.major.width':    0.8,
    'xtick.major.size':     3,
    'ytick.major.size':     3,
    'xtick.direction':      'out',
    'ytick.direction':      'out',
    # ── Lines ────────────────────────────────────────────────────────
    'lines.linewidth':      1.0,
    # ── Save settings ─────────────────────────────────────────────────
    'figure.dpi':           150,
    'savefig.dpi':          300,
    'savefig.bbox':         'tight',
    'savefig.pad_inches':   0.05,
})
print(f'Matplotlib style applied (font: {_use_font})')


---
## 1. Configuration

Set the paths and key parameters here. The most important setting is `SIM_DIR`: point this at the folder created by `brain_act_cached_analysis_legacy_pooled_per_scenario.ipynb` (typically `notebooks/outputs/ba_dual2min/sims/`). If the directory does not exist, the notebook will generate synthetic data automatically.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
#  CONFIGURATION  —  edit these to match your data layout
# ═══════════════════════════════════════════════════════════════════════════════

# ── Route selection ───────────────────────────────────────────────────────────
# Must match the route that was run in brain_act_simulations_v2.ipynb:
#   "private_only"  →  only α=0 (Route A)
#   "full_gamut"    →  all 5 noise scenarios (Route B)
ROUTE_SELECT     = "private_only"

# Which scenarios to load (None = all available in this route)
SCENARIO_FILTER  = None   # e.g. ["private_alpha0"] to restrict further

# ── Data directories ──────────────────────────────────────────────────────────
# brain_act_simulations_v2.ipynb writes to:
#   ba_sim_v2/<route>/sims/<scenario>/<cohort>/<subject_id>/seed_NNN.npz
#   ba_sim_v2/<route>/sims_pci/<scenario>/<cohort>/<subject_id>/trial_NNN.npz
SIM_DIR          = PROJECT_ROOT / 'notebooks' / 'outputs' / 'ba_sim_v2' / ROUTE_SELECT / 'sims'
SIM_PCI_DIR      = PROJECT_ROOT / 'notebooks' / 'outputs' / 'ba_sim_v2' / ROUTE_SELECT / 'sims_pci'

# Structural connectivity dataset (same root as simulation notebook)
DATASET_ROOT     = PROJECT_ROOT / 'data' / 'doc_patients_new_data' / 'converted_structural'

# Output directory for figures produced by this notebook
FIG_DIR = PROJECT_ROOT / 'notebooks' / 'outputs' / 'ba_full_v2' / 'figs'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Cohort → display-condition mapping ───────────────────────────────────────
# Maps the cohort folder name (as saved by the simulation notebook) to a short
# display label used throughout this analysis notebook.
COHORT_TO_CONDITION = {
    "control": "CNT",
    "emcs":    "EMCS",
    "mcs":     "MCS",
    "uws":     "UWS",
    "coma":    "COMA",
}
# Display order for plots (lowest → highest consciousness level)
CONDITION_ORDER  = ["COMA", "UWS", "MCS", "EMCS", "CNT"]
# Only include conditions that are present in the dataset (populated after loading)
CONDITIONS       = CONDITION_ORDER   # will be pruned in the loader cell

# ── Wes Anderson colour palette ───────────────────────────────────────────────
# Inspired by "The Grand Budapest Hotel" + "Moonrise Kingdom"
# Ordered low→high consciousness: deep indigo → dusty mauve → sienna → amber → sage
WES_PALETTE = {
    "COMA":  "#3B4A6B",   # midnight slate
    "UWS":   "#8B6B8B",   # dusty mauve
    "MCS":   "#C5622F",   # burnt sienna
    "EMCS":  "#E8B56D",   # warm amber
    "CNT":   "#5B8A72",   # forest sage
}
COND_COLORS = WES_PALETTE  # alias used throughout notebook

# Sedation stratification ─────────────────────────────────────────────────────
# Default mapping: COMA + UWS → sedated (often on ICU sedation); rest → unsedated.
# Override with actual clinical metadata if available (add a 'sedated' column to
# your subject index and load it in Cell 6).
SEDATION_MAP = {
    "COMA":  "Sedated",
    "UWS":   "Sedated",
    "MCS":   "Unsedated",
    "EMCS":  "Unsedated",
    "CNT":   "Unsedated",
}
SEDATION_COLORS = {
    "Sedated":   "#7B6F8E",   # dusty violet
    "Unsedated": "#5B8A72",   # sage green
}

# Brain-state colour palette (5 states low→high SC-FC coupling)
STATE_PALETTE = ["#3B4A6B", "#8B6B8B", "#C5622F", "#E8B56D", "#5B8A72"]

# ── Brain-state clustering ────────────────────────────────────────────────────
N_STATES           = 5
RANDOM_SEED_KMEANS = 42
N_INIT_KMEANS      = 10
MAX_ITER_KMEANS    = 300

# ── Firing-rate preprocessing ─────────────────────────────────────────────────
# NOTE: simulations are saved POST-transient — no further transient stripping needed here.
DT_MS              = 5.0            # bin width of rate monitor output (ms)
TRANSIENT_MS       = 0.0            # simulations are saved post-transient; no further stripping needed, matches RATE_MONITOR_PERIOD_MS
BANDPASS_RATE_HZ   = (2.0, 80.0)   # 2–80 Hz: AdEx adaptation + gamma; valid at 200 Hz
FILTER_ORDER_RATE  = 4

# ── BOLD preprocessing ────────────────────────────────────────────────────────
TR_SECONDS         = 2.4            # fMRI repetition time (s)
BANDPASS_BOLD_HZ   = (0.01, 0.20)
FILTER_ORDER_BOLD  = 3

# ── PCI parameters — matching TVBSim defaults (n_trials now 10) ───────────────
N_TRIALS_PCI       = 100    # 100 independent trials per subject — must match N_TRIALS_PCI in sim notebook
T_ANALYSIS_MS_PCI  = 300.0  # one-sided analysis window (ms) — TVBSim t_analysis=300
NSHUFFLES_PCI      = 10     # surrogate shuffles for binarisation (TVBSim default)
PERCENTILE_PCI     = 100.0  # threshold percentile (TVBSim convention)

# Maximum subjects per condition to load (None = load all)
N_SUBJECTS_MAX     = None

print("Configuration loaded.")
print(f"  SIM_DIR:     {SIM_DIR}")
print(f"  SIM_PCI_DIR: {SIM_PCI_DIR}")
print(f"  DT_MS:       {DT_MS} ms  ({1000/DT_MS:.0f} Hz rate monitor)")
print(f"  N_TRIALS_PCI: {N_TRIALS_PCI}")


---
## 2. Data Loading — Simulations and Their Structure

### What are we simulating?

The Brain-Act pipeline uses the **Zerlaut/AdEx mean-field (MF) model** embedded in a whole-brain TVB (The Virtual Brain) framework. The model describes each of the 68 Desikan-Killiany brain regions as a local population of excitatory (E) and inhibitory (I) neurons connected via empirical structural connectivity (SC) from diffusion MRI tractography.

Each brain region produces:
- **Excitatory firing rate** *E(t)* — stored in NPZ as **kHz** (TVB/AdEx convention). The analysis notebook converts to **Hz** on load (`rate × 1000`) for physiological labelling.
- **BOLD signal** — computed via a balloon-Windkessel haemodynamic model that converts E(t) into the 1-2% fluctuations measured by fMRI

### Disorders of Consciousness (DoC) modelling

Three conditions are modelled:

| Condition | Full name | Model mechanism |
|-----------|-----------|------------------|
| **CNT** | Controls / Wakefulness | High excitability, strong long-range coupling |
| **MCS** | Minimally Conscious State | Reduced excitability or partial SC damage |
| **UWS** | Unresponsive Wakefulness Syndrome | Severely reduced excitability / heavy SC damage |

### Two temporal scales

The simulation captures two distinct timescales:

1. **Firing rate (5 ms bins, 2–80 Hz)**: The fast neural signal. Captures:
   - γ-band (30–80 Hz): local E/I balance, synaptic resonance
   - α/β (8–30 Hz): thalamo-cortical loops
   - δ/θ (2–8 Hz): adaptation-driven slow oscillations (τ_w = 500 ms)

2. **BOLD (2.4 s TR, 0.01–0.20 Hz)**: The slow haemodynamic echo. The HRF peaks at ≈6 s and acts as a low-pass filter — it cannot follow neural events faster than ≈0.2 Hz.

> **Crucial implication for analysis:** Brain states and LZc computed on firing rates reflect *fast neural dynamics*. The same measures on BOLD reflect *slow infra-slow co-activation patterns*. These are complementary, not redundant.

In [ ]:
# ────────────────────────────────────────────────────────────────────────────
#  Load simulations produced by brain_act_simulations_v2.ipynb
#  NPZ layout (spontaneous):
#    time_rate_ms  (T_rate,)        — rate monitor time axis
#    rate          (T_rate, N_reg)  — excitatory firing rates (spikes/s)
#    time_bold_ms  (T_bold,)        — BOLD monitor time axis  [optional]
#    bold          (T_bold, N_reg)  — BOLD signal             [optional]
#    region_labels (N_reg,)
#    noise_alpha, seed, transient_ms, rate_monitor_period_ms
#  Both domains are already POST-transient (transient stripped in simulation).
#
#  SC is loaded from the structural dataset (not stored in NPZ to save space).
# ────────────────────────────────────────────────────────────────────────────

try:
    from tvbtoolkit.datasets.brain_act import load_subject_structural
    _HAS_TVBTOOLKIT = True
except ImportError:
    _HAS_TVBTOOLKIT = False
    print("WARNING: tvbtoolkit not importable — SC will be set to identity matrices.")


def _load_sc(cohort: str, subject_id: str) -> np.ndarray:
    """Load SC matrix with the same damage-masking and normalisation as the simulations.

    Applies:
      1. Symmetrisation and zero-diagonal (via load_subject_structural)
      2. Damage parity: for patient cohorts, zero tract-length mismatches are
         propagated to SC (absent connections zeroed in l where c==0)
         — here we just need the SC mask, so we zero SC where l==0 after parity
      3. Max-normalisation over surviving edges
    This mirrors _apply_damage_parity() in brain_act_dual_domain_parallel.py.
    """
    if not _HAS_TVBTOOLKIT or not DATASET_ROOT.exists():
        return np.eye(10)   # fallback identity
    try:
        c, l, _atlas, _meta = load_subject_structural(
            subject_id=subject_id,
            cohort=cohort,
            dataset_root=DATASET_ROOT,
            validate=True,
            enforce_symmetry=True,
            zero_diagonal=True,
            nonfinite="raise",
        )
        c = np.asarray(c, dtype=float)
        l = np.asarray(l, dtype=float)
        np.fill_diagonal(c, 0.0)
        np.fill_diagonal(l, 0.0)
        # Damage parity for patient cohorts
        if cohort.lower() in {"coma", "uws", "mcs", "emcs"}:
            mismatch = (c == 0.0) & (l != 0.0)
            if np.any(mismatch):
                l[mismatch] = 0.0
        # Max-normalise over surviving edges
        c_max = float(np.max(c))
        if c_max > 0.0:
            c = c / c_max
        return c
    except Exception as exc:
        print(f"  SC load failed ({cohort}/{subject_id}): {exc}")
        return np.eye(10)


def _load_from_sim_dir(sim_dir: Path) -> "dict[str, list[dict]]":
    """Walk the ba_sim_v2 directory tree and load all subject NPZ files.

    Expected layout:
      sim_dir / <scenario_key> / <cohort> / <subject_id> / seed_NNN.npz

    Returns a mapping  condition_label → list of subject data dicts.
    Each dict contains the keys expected by downstream analysis cells:
      rate_time, rate, bold_time, bold, sc, condition, cohort, subject_id,
      scenario_key, synthetic=False
    """
    data_out: dict[str, list[dict]] = {c: [] for c in CONDITION_ORDER}

    if not sim_dir.exists():
        print(f"SIM_DIR not found: {sim_dir}")
        return data_out

    for scenario_path in sorted(sim_dir.iterdir()):
        if not scenario_path.is_dir():
            continue
        scenario_key = scenario_path.name
        if SCENARIO_FILTER is not None and scenario_key not in SCENARIO_FILTER:
            continue

        for cohort_path in sorted(scenario_path.iterdir()):
            if not cohort_path.is_dir():
                continue
            cohort = cohort_path.name
            cond = COHORT_TO_CONDITION.get(cohort)
            if cond is None:
                continue   # unknown cohort — skip

            for subj_path in sorted(cohort_path.iterdir()):
                if not subj_path.is_dir():
                    continue
                subject_id = subj_path.name

                if N_SUBJECTS_MAX is not None and len(data_out[cond]) >= N_SUBJECTS_MAX:
                    continue

                npz_files = sorted(subj_path.glob("seed_*.npz"))
                if not npz_files:
                    continue   # no NPZ yet (simulation not yet run)

                # Resolve symlinks transparently
                npz_path = npz_files[0].resolve() if npz_files[0].is_symlink() else npz_files[0]
                try:
                    d = np.load(npz_path, allow_pickle=True)
                except Exception as exc:
                    print(f"  Could not load {npz_path}: {exc}")
                    continue

                # ── Rate domain — AdEx E stored in kHz; convert to Hz here ──
                rate_time = np.asarray(d["time_rate_ms"], dtype=float)
                rate      = np.asarray(d["rate"],         dtype=float) * 1e3   # kHz → Hz

                # ── BOLD domain (optional — may not be present) ───────────
                bold_time_key = "time_bold_ms" if "time_bold_ms" in d else None
                bold_key      = "bold"         if "bold"         in d else None
                bold_time = np.asarray(d[bold_time_key], dtype=float) if bold_time_key else np.array([])
                bold      = np.asarray(d[bold_key],      dtype=float) if bold_key      else np.empty((0, rate.shape[1]))

                # ── Structural connectivity ───────────────────────────────
                sc = _load_sc(cohort, subject_id)

                data_out[cond].append({
                    "rate_time":   rate_time,
                    "rate":        rate,
                    "bold_time":   bold_time,
                    "bold":        bold,
                    "sc":          sc,
                    "condition":   cond,
                    "cohort":      cohort,
                    "subject_id":  subject_id,
                    "scenario_key": scenario_key,
                    "synthetic":   False,
                })

    return data_out


# ── Synthetic fallback (used when sim_dir is absent) ─────────────────────────
def _generate_synthetic_data(
    condition: str,
    n_regions: int = 10,
    t_rate_ms: float = 30_000.0,
    dt_ms: float = DT_MS,
    rng: "np.random.Generator | None" = None,
) -> dict:
    """Generate realistic synthetic firing-rate + BOLD data for one subject.

    CNT: strong gamma (~40 Hz), complex spatial coupling → high complexity
    MCS: moderate oscillations, weaker coupling
    UWS: near-noise activity, minimal structure → low complexity
    """
    if rng is None:
        rng = np.random.default_rng()
    n_bins = int(t_rate_ms / dt_ms)
    t_rate = np.arange(n_bins) * dt_ms

    params = {
        "CNT":  dict(f_osc=40.0, amp=8.0, noise=2.0, coupling=0.6, baseline=5.0),
        "MCS":  dict(f_osc=20.0, amp=3.5, noise=2.5, coupling=0.3, baseline=3.0),
        "UWS":  dict(f_osc=5.0,  amp=0.8, noise=3.0, coupling=0.1, baseline=1.5),
        "EMCS": dict(f_osc=30.0, amp=5.5, noise=2.2, coupling=0.5, baseline=4.0),
        "COMA": dict(f_osc=3.0,  amp=0.5, noise=3.5, coupling=0.05, baseline=1.0),
    }
    p = params.get(condition, params["MCS"])

    sc = rng.uniform(0, 1, (n_regions, n_regions))
    sc = (sc + sc.T) / 2
    sc[sc < 0.5] = 0.0
    np.fill_diagonal(sc, 0.0)
    sc /= sc.max() + 1e-9

    omega = 2 * np.pi * p["f_osc"] / 1000.0
    phase_offsets = rng.uniform(0, 2 * np.pi, n_regions)
    rate = np.zeros((n_bins, n_regions))
    for i in range(n_regions):
        osc    = p["amp"] * np.sin(omega * t_rate + phase_offsets[i])
        shared = p["coupling"] * p["amp"] * np.sin(0.3 * omega * t_rate)
        noise  = p["noise"] * rng.standard_normal(n_bins)
        rate[:, i] = np.maximum(0.0, p["baseline"] + osc + shared + noise)

    bold_stride = max(1, int(TR_SECONDS * 1000 / dt_ms))
    bold_idx    = np.arange(0, n_bins, bold_stride)
    bold_time   = t_rate[bold_idx]
    kern_len    = min(bold_stride * 2, n_bins // 2)
    kern        = np.ones(kern_len) / kern_len
    bold_smeared = np.apply_along_axis(lambda x: np.convolve(x, kern, "same"), 0, rate)
    bold        = bold_smeared[bold_idx, :]
    bold       += 0.1 * rng.standard_normal(bold.shape)
    bold        = (bold - bold.mean(0, keepdims=True)) / (bold.std(0, keepdims=True) + 1e-9)

    return {
        "rate_time":   t_rate,
        "rate":        rate,
        "bold_time":   bold_time,
        "bold":        bold,
        "sc":          sc,
        "condition":   condition,
        "cohort":      condition.lower(),
        "subject_id":  "synthetic",
        "scenario_key": "private_alpha0",
        "synthetic":   True,
    }


# ── Load or generate ──────────────────────────────────────────────────────────
N_SUBJECTS_SYNTHETIC = 5   # subjects per condition when falling back to synthetic

data: dict[str, list[dict]] = _load_from_sim_dir(SIM_DIR)

# Determine which conditions actually have data
CONDITIONS = [c for c in CONDITION_ORDER if data.get(c)]

USE_SYNTHETIC = len(CONDITIONS) == 0
if USE_SYNTHETIC:
    print("Simulation directory empty or not found — generating synthetic data.")
    data = {c: [] for c in CONDITION_ORDER}
    for cond in ["CNT", "MCS", "UWS"]:
        for s in range(N_SUBJECTS_SYNTHETIC):
            rng_s = np.random.default_rng(seed=abs(hash(cond + str(s))) % (2**31))
            data[cond].append(_generate_synthetic_data(cond, n_regions=10, rng=rng_s))
    CONDITIONS = [c for c in CONDITION_ORDER if data.get(c)]

print(f"\nLoaded conditions: {CONDITIONS}")
print(f"Data source: {'SYNTHETIC (fallback)' if USE_SYNTHETIC else str(SIM_DIR)}")
for cond in CONDITIONS:
    n = len(data[cond])
    if n:
        ex = data[cond][0]
        r_shape = np.asarray(ex["rate"]).shape
        b_shape = np.asarray(ex["bold"]).shape
        print(f"  {cond:5s}: {n} subjects  |  rate {r_shape}  bold {b_shape}")


---
## 3. Data Exploration — Raw Signals

Before any analysis it is essential to look at the raw signals. We want to confirm:
- Firing rates are in a physiological range (0–100 Hz for AdEx MF)
- BOLD fluctuations are small (<5%) and slow as expected
- The three conditions show visibly different dynamics

In [ ]:
n_cond = len(CONDITIONS)
fig, axes = plt.subplots(2, n_cond, figsize=(5 * n_cond, 6), sharex=False)
fig.suptitle('Raw simulation outputs — one subject per condition', fontsize=13, fontweight='bold')

axes = np.array(axes).reshape(2, n_cond)
for col, cond in enumerate(CONDITIONS):
    if not data[cond]:
        continue
    ex = data[cond][0]
    rate = np.asarray(ex['rate'])       # (time, regions)
    bold = np.asarray(ex['bold'])       # (time, regions)
    t_rate_s = np.asarray(ex['rate_time']) / 1000.0
    t_bold_s = np.asarray(ex['bold_time']) / 1000.0
    n_regions = rate.shape[1]
    color = COND_COLORS[cond]
    
    # Top row: firing rates
    ax = axes[0, col]
    show_n = min(5, n_regions)
    for r in range(show_n):
        alpha = 0.6 if r > 0 else 1.0
        ax.plot(t_rate_s[:3000], rate[:3000, r], lw=0.6, alpha=alpha, color=color)
    ax.set_title(f'{cond}', fontsize=12, fontweight='bold', color=color)
    ax.set_ylabel('Firing rate (Hz)' if col == 0 else '')
    ax.set_xlabel('Time (s)')
    ax.text(0.02, 0.95, f'Firing rates ({show_n}/{n_regions} regions)', 
            transform=ax.transAxes, fontsize=8, va='top')
    
    # Bottom row: BOLD
    ax = axes[1, col]
    for r in range(show_n):
        alpha = 0.6 if r > 0 else 1.0
        ax.plot(t_bold_s, bold[:, r], lw=0.9, alpha=alpha, color=color)
    ax.set_ylabel('BOLD (z-score)' if col == 0 else '')
    ax.set_xlabel('Time (s)')
    ax.text(0.02, 0.95, f'BOLD ({show_n}/{n_regions} regions)', 
            transform=ax.transAxes, fontsize=8, va='top')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig00_raw_signals.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 4. Spectral Validity Check

### Why we must check spectral content before phase analysis

The Phase-Coherence Brain States analysis depends critically on the **Hilbert transform** to extract instantaneous phase. The Hilbert transform is only a meaningful estimator of instantaneous phase when the signal is **narrowband** — i.e., when it contains a single dominant oscillatory component.

#### The "twelve overlapping clocks" problem

Imagine you are asked to tell the time from a room full of twelve clocks, all ticking at slightly different rates and with their hands superimposed on a single display. You cannot tell what time any individual clock shows. This is analogous to applying the Hilbert transform to a **broadband** signal: the analytic phase becomes a mixture of contributions from all frequency bands simultaneously, producing what engineers call **phase slippage** — rapid, meaningless jumps in the estimated phase that have nothing to do with the underlying oscillatory dynamics.

#### Solution: narrowband filtering first

By applying a bandpass filter before the Hilbert transform, we isolate a single dominant frequency band. The phase of the filtered signal now reflects the genuine oscillatory phase of the selected rhythm:

$$\phi_i(t) = \arg\left( \mathcal{H}\left[x_i^{\text{filtered}}(t)\right] \right)$$

#### Spectral checks we perform

1. **Power Spectral Density (PSD)**: Welch's method estimates the average power at each frequency. We look for a clear spectral peak within the analysis band.

2. **Spectral SNR**: In-band peak power / out-of-band median power. SNR > 3 is considered adequate for phase analysis.

3. **Kuramoto order parameter** after filtering:
$$R(t) = \left| \frac{1}{N} \sum_{i=1}^{N} e^{j\phi_i(t)} \right|$$
   $R \approx 0$: random phases (no synchrony). $R \approx 1$: all regions phase-locked. We require $R > 0.05$ as a minimum sign of coherent oscillation.

In [ ]:
n_cond = len(CONDITIONS)
fig, axes = plt.subplots(1, n_cond, figsize=(5 * n_cond, 4), sharey=True)
fig.suptitle('Power Spectral Density — Firing Rates (Welch, averaged over regions)', fontsize=12)

validity_results = {}
axes = np.array(axes).reshape(n_cond)
for col, cond in enumerate(CONDITIONS):
    if not data[cond]:
        continue
    ex = data[cond][0]
    rate = np.asarray(ex['rate'], dtype=float)
    ax = axes[col]
    
    # Compute PSD
    psd_result = psd_per_region(rate, DT_MS)
    freqs = psd_result.frequencies          # Hz
    power = psd_result.power                # (n_freqs, n_regions)
    power_mean = np.mean(power, axis=1)     # average over regions
    
    # Shading: analysis band
    band_mask = (freqs >= BANDPASS_RATE_HZ[0]) & (freqs <= BANDPASS_RATE_HZ[1])
    ax.semilogy(freqs, power_mean, color=COND_COLORS[cond], lw=1.5, label='Mean PSD')
    ax.fill_between(freqs, power_mean, where=band_mask, alpha=0.25,
                    color=COND_COLORS[cond], label=f'{BANDPASS_RATE_HZ[0]}–{BANDPASS_RATE_HZ[1]} Hz band')
    ax.axvline(BANDPASS_RATE_HZ[0], ls='--', lw=0.8, color='grey')
    ax.axvline(BANDPASS_RATE_HZ[1], ls='--', lw=0.8, color='grey')
    ax.set_title(cond, fontsize=11, fontweight='bold', color=COND_COLORS[cond])
    ax.set_xlabel('Frequency (Hz)')
    if col == 0:
        ax.set_ylabel('Power (arbitrary units)')
    ax.legend(fontsize=7)
    ax.set_xlim(0, 100)
    
    # Run validity check
    validity = phase_coherence_validity(rate, DT_MS, BANDPASS_RATE_HZ,
                                         filter_order=FILTER_ORDER_RATE)
    validity_results[cond] = validity
    info = f'Kuramoto R̄={validity.kuramoto_order:.3f}'
    ax.text(0.97, 0.95, info, transform=ax.transAxes, fontsize=8,
            ha='right', va='top',
            bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))
    if validity.warnings:
        print(f'  [{cond}] Validity warnings: {validity.warnings}')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig01_spectral_validity.png', dpi=150, bbox_inches='tight')
plt.show()

print('Spectral validity summary:')
for cond, v in validity_results.items():
    pct_peaks = np.mean(v.has_spectral_peak) * 100
    print(f'  {cond}:  Kuramoto={v.kuramoto_order:.3f}  '
          f'Peak in band={pct_peaks:.0f}% of regions  '
          f'Mean amp={np.mean(v.mean_amplitude):.4f}')

---
## 5. Phase-Coherence Brain States

### 5.1 Theory — From signals to brain states

#### Step 1: Preprocessing

**For firing rates** (pipeline `'firing_rate'`):
1. Discard initial transient (first 500 ms)
2. Z-score each region over time
3. Apply 4th-order Butterworth bandpass filter: 2–80 Hz

**For BOLD** (pipeline `'brain_act_legacy'`):
1. Z-score each ROI (μ→0, σ→1)
2. Subtract the cross-ROI mean at each time point (remove global signal)
3. Apply 3rd-order Butterworth bandpass: 0.01–0.20 Hz

#### Step 2: Instantaneous phase via Hilbert transform

For each region $i$ and time point $t$, the analytic signal is:
$$z_i(t) = x_i(t) + j \cdot \mathcal{H}[x_i(t)]$$

where $\mathcal{H}[\cdot]$ is the Hilbert transform. The **instantaneous phase** is:
$$\phi_i(t) = \arg\left(z_i(t)\right) \in [-\pi, \pi]$$

#### Step 3: Phase-coherence patterns

At each time step $t$, we compute the cosine of the phase difference for every pair of regions $(i, j)$ with $i < j$:

$$P_{ij}(t) = \cos\left(\phi_i(t) - \phi_j(t)\right)$$

$P_{ij}(t) = +1$ means regions $i$ and $j$ are perfectly in phase (maximum cooperation).
$P_{ij}(t) = -1$ means they are perfectly anti-phase (maximum competition).

This gives us a vector $\mathbf{p}(t) \in [-1,1]^{N_\text{edges}}$ at each time step, where $N_\text{edges} = N(N-1)/2$ for $N$ regions.

#### Step 4: Global synchrony — the Kuramoto order parameter

At each time step, we also compute:
$$R(t) = \left| \frac{1}{N} \sum_{i=1}^{N} e^{j\phi_i(t)} \right|$$

This scalar tracks how synchronized all regions are at each moment.

#### Step 5: K-means clustering

We collect all time-step vectors $\{\mathbf{p}(t)\}$ and cluster them using **K-means** ($K=5$):
$$\arg\min_{\{\mathbf{c}_k\}} \sum_t \min_k \|\mathbf{p}(t) - \mathbf{c}_k\|^2$$

Each cluster centroid $\mathbf{c}_k$ is a **brain state** — a characteristic spatial pattern of inter-regional phase relationships.

#### Step 6: Centroid sorting by SC-FC coupling (SFC sort)

To make brain states comparable across subjects and studies, centroids are sorted by their Pearson correlation with the upper triangle of the structural connectivity (SC) matrix — ordered from least SC-like (state 0) to most SC-like (state $K-1$). This matches Brain-Act legacy convention.

#### Derived metrics

- **Occupancy** $O_k$: fraction of time in state $k$ → how often the brain visits each state
- **Transition matrix** $T_{kl}$: probability of going from state $k$ to state $l$ → state dynamics
- **SC-FC coupling**: Pearson $r$ between state centroids and SC upper triangle → measures how much network structure drives synchrony patterns

In [ ]:
# ── Brain States on Firing Rates ──────────────────────────────────────────────
print('Computing brain states on firing rates (pipeline=firing_rate)...')

bs_rate: dict[str, list[BrainStateSummary]] = {c: [] for c in CONDITIONS}
sfc_rate: dict[str, list] = {c: [] for c in CONDITIONS}  # SC-FC coupling per subject

for cond in CONDITIONS:
    for s, subj in enumerate(data[cond]):
        rate = np.asarray(subj['rate'], dtype=float)   # (time, regions)
        sc   = np.asarray(subj['sc'],   dtype=float)
        
        summary = summarize_brain_states(
            rate,
            n_states=N_STATES,
            pipeline='firing_rate',
            bandpass_hz=BANDPASS_RATE_HZ,
            filter_order=FILTER_ORDER_RATE,
            dt_ms=DT_MS,
            transient_ms=TRANSIENT_MS,
            random_seed=RANDOM_SEED_KMEANS,
            n_init=N_INIT_KMEANS,
            max_iter=MAX_ITER_KMEANS,
        )
        
        # Sort centroids by SC-FC coupling (Brain-Act legacy convention)
        centers_s, labels_s, order_s, sfc_s = sfc_sort_centroids(
            summary.centers, summary.labels, sc
        )
        from tvbtoolkit.analysis.brain_states import BrainStateSummary
        from tvbtoolkit.analysis.brain_states import (
            _compute_occupancy, _compute_transition_matrix
        )
        n_eff = centers_s.shape[0]
        summary_sorted = BrainStateSummary(
            labels=labels_s,
            centers=centers_s,
            occupancy=_compute_occupancy(labels_s, n_eff),
            transition_matrix=_compute_transition_matrix(labels_s, n_eff),
            global_synchrony=summary.global_synchrony,
            edge_index_i=summary.edge_index_i,
            edge_index_j=summary.edge_index_j,
        )
        bs_rate[cond].append(summary_sorted)
        sfc_rate[cond].append(sfc_s)  # sorted SFC values (ascending)
        
    print(f'  {cond}: done  (occupancy example: '
          f'{np.round(bs_rate[cond][0].occupancy, 2)})')

print('\nBrain states (firing rates): complete.')

In [ ]:
# ── Brain States on BOLD ──────────────────────────────────────────────────────
print('Computing brain states on BOLD (pipeline=brain_act_legacy)...')

bs_bold: dict[str, list[BrainStateSummary]] = {c: [] for c in CONDITIONS}
sfc_bold: dict[str, list] = {c: [] for c in CONDITIONS}  # SC-FC coupling per subject

for cond in CONDITIONS:
    for s, subj in enumerate(data[cond]):
        bold = np.asarray(subj['bold'], dtype=float)
        sc   = np.asarray(subj['sc'],  dtype=float)
        
        # BOLD data may be too short for filtfilt with this band — guard gracefully
        if bold.shape[0] < 30:
            # Insufficient BOLD samples — skip and store empty
            bs_bold[cond].append(None)
            sfc_bold[cond].append(None)
            continue
        
        try:
            summary = summarize_brain_states(
                bold,
                n_states=N_STATES,
                pipeline='brain_act_legacy',
                tr_seconds=TR_SECONDS,
                bandpass_hz=BANDPASS_BOLD_HZ,
                filter_order=FILTER_ORDER_BOLD,
                random_seed=RANDOM_SEED_KMEANS,
                n_init=N_INIT_KMEANS,
                max_iter=MAX_ITER_KMEANS,
            )
            centers_s, labels_s, _ord_b, sfc_b = sfc_sort_centroids(
                summary.centers, summary.labels, sc
            )
            n_eff = centers_s.shape[0]
            summary_sorted = BrainStateSummary(
                labels=labels_s,
                centers=centers_s,
                occupancy=_compute_occupancy(labels_s, n_eff),
                transition_matrix=_compute_transition_matrix(labels_s, n_eff),
                global_synchrony=summary.global_synchrony,
                edge_index_i=summary.edge_index_i,
                edge_index_j=summary.edge_index_j,
            )
            bs_bold[cond].append(summary_sorted)
            sfc_bold[cond].append(sfc_b)
        except Exception as exc:
            print(f'  Warning [{cond} subj {s}]: {exc}')
            bs_bold[cond].append(None)
            sfc_bold[cond].append(None)

    valid = sum(1 for x in bs_bold[cond] if x is not None)
    print(f'  {cond}: {valid}/{len(data[cond])} subjects computed')

print('\nBrain states (BOLD): complete.')

In [ ]:
# ── Visualise Brain States ────────────────────────────────────────────────────
# We show centroids, occupancy and transition matrix for one subject per condition.

def plot_brain_state_summary(
    summary: BrainStateSummary,
    n_regions: int,
    title: str,
    color: str,
    fig=None, axes=None, row=0
):
    """Plot centroid matrices (as heatmaps), occupancy bar, transition heatmap, synchrony trace."""
    if summary is None:
        return
    K = summary.centers.shape[0]
    centroids = centers_to_matrices(summary, n_regions)  # (K, R, R)
    
    ax_occ = axes[row, 0]
    occ = summary.occupancy
    bars = ax_occ.bar(range(K), occ * 100, color=color, alpha=0.75, edgecolor='black', lw=0.5)
    ax_occ.set_xlabel('Brain state')
    ax_occ.set_ylabel('Occupancy (%)')
    ax_occ.set_title(f'{title}\nOccupancy', fontsize=9)
    ax_occ.set_xticks(range(K))
    ax_occ.set_ylim(0, 55)

    ax_tm = axes[row, 1]
    tm = summary.transition_matrix
    im = ax_tm.imshow(tm, vmin=0, vmax=1, cmap='Blues', aspect='auto')
    ax_tm.set_title(f'{title}\nTransition matrix', fontsize=9)
    ax_tm.set_xlabel('To state')
    ax_tm.set_ylabel('From state')
    plt.colorbar(im, ax=ax_tm, fraction=0.046, pad=0.04)

    ax_sync = axes[row, 2]
    t_valid = np.arange(len(summary.global_synchrony)) * DT_MS / 1000.0
    ax_sync.plot(t_valid[:3000], summary.global_synchrony[:3000], lw=0.6, color=color, alpha=0.8)
    ax_sync.set_xlabel('Time (s)')
    ax_sync.set_ylabel('Kuramoto R(t)')
    ax_sync.set_ylim(0, 1)
    ax_sync.set_title(f'{title}\nGlobal synchrony', fontsize=9)

    # Centroid grid (first 3 centroids)
    for k in range(min(K, 3)):
        if k + 3 >= axes.shape[1]:
            break
        ax_c = axes[row, k + 3]
        im_c = ax_c.imshow(centroids[k], vmin=-1, vmax=1, cmap='RdBu_r', aspect='auto')
        ax_c.set_title(f'State {k}', fontsize=8)
        ax_c.axis('off')


n_regions_ex = np.asarray(data[CONDITIONS[0]][0]['rate']).shape[1]
n_cond = len(CONDITIONS)
n_cols = 3 + min(N_STATES, 3)
fig, axes = plt.subplots(n_cond, n_cols, figsize=(4 * n_cols, 3 * n_cond))
axes = np.array(axes).reshape(n_cond, n_cols)
fig.suptitle('Brain State Analysis — Firing Rates (one subject per condition)', fontsize=12, fontweight='bold')

for row, cond in enumerate(CONDITIONS):
    if not bs_rate[cond]:
        continue
    plot_brain_state_summary(
        bs_rate[cond][0],
        n_regions=n_regions_ex,
        title=cond,
        color=COND_COLORS[cond],
        fig=fig, axes=axes, row=row
    )

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig02_brain_states_rate.png', dpi=150, bbox_inches='tight')
plt.show()

print('Brain state figures saved.')

In [ ]:
# ── Figure: State Occupancy vs SC-FC Coupling — Firing Rates AND BOLD ─────────
# Two panels: (A) firing-rate brain states  (B) BOLD brain states
# For each subject × state: x = SC-FC coupling (Pearson r), y = occupancy (%)
# Per-cluster regression with 95% CI and annotated r/p values.

from scipy import stats as _stats
import matplotlib.patches as _mpatches

def _occ_sfc_panel(ax, bs_dict, sfc_dict, title, K=N_STATES):
    """Draw occupancy vs SFC scatter + per-cluster regression onto ax."""
    sfc_arr, occ_arr, cond_arr = [], [], []

    for cond in CONDITIONS:
        for bs, sfc in zip(bs_dict.get(cond, []), sfc_dict.get(cond, [])):
            if bs is None or sfc is None:
                continue
            occ_arr.append(bs.occupancy * 100)
            sfc_arr.append(sfc)
            cond_arr.append(cond)

    if not sfc_arr:
        ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title, fontsize=8, loc='left')
        return 0

    sfc_arr = np.array(sfc_arr)   # (N_subj, K)
    occ_arr = np.array(occ_arr)   # (N_subj, K)
    N = sfc_arr.shape[0]
    K_eff = min(sfc_arr.shape[1], K, len(STATE_PALETTE))

    for k in range(K_eff):
        x = sfc_arr[:, k]
        y = occ_arr[:, k]
        col = STATE_PALETTE[k]

        # Subject dots coloured by condition
        for cond in CONDITIONS:
            mask = np.array([c == cond for c in cond_arr])
            if np.any(mask):
                ax.scatter(x[mask], y[mask], color=col, s=16,
                           marker='o', alpha=0.65,
                           edgecolors='white', linewidths=0.3, zorder=3)

        # Regression
        if len(x) > 2:
            sl, ic, r, p, _ = _stats.linregress(x, y)
            x_line = np.linspace(x.min(), x.max(), 80)
            sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            ax.plot(x_line, sl * x_line + ic, color=col, lw=1.8, zorder=4,
                    label=f'S{k}  r={r:.2f}{sig}  p={p:.3f}')
            # 95% CI
            n_ = len(x)
            se_ = np.sqrt(np.sum((y - (sl*x + ic))**2) / (n_-2)) * \
                  np.sqrt(1/n_ + (x_line - x.mean())**2 / np.sum((x - x.mean())**2))
            t_ = _stats.t.ppf(0.975, df=n_-2)
            ax.fill_between(x_line, sl*x_line+ic - t_*se_,
                            sl*x_line+ic + t_*se_, color=col, alpha=0.10, zorder=2)

    ax.set_xlabel('SC-FC coupling (Pearson r)', fontsize=8)
    ax.set_ylabel('State occupancy (%)', fontsize=8)
    ax.set_title(title, fontsize=8, loc='left', fontweight='bold')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=6, frameon=False, bbox_to_anchor=(1.01, 1), loc='upper left',
              title='State (low→high SC-FC)', title_fontsize=6)
    ax.text(0.02, 0.97, 'States sorted: low SC-FC → high SC-FC',
            transform=ax.transAxes, fontsize=6, va='top', color='#666666')
    return N

# ── Build figure ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(7.08, 3.2))

N_rate = _occ_sfc_panel(axes[0], bs_rate, sfc_rate,
                         'A  Occupancy vs SC-FC  (firing rates)')
N_bold = _occ_sfc_panel(axes[1], bs_bold, sfc_bold,
                         'B  Occupancy vs SC-FC  (BOLD)')

# Condition legend (shared)
leg_handles = [_mpatches.Patch(color=COND_COLORS[c], alpha=0.85, label=c)
               for c in CONDITIONS if c in COND_COLORS]
fig.legend(handles=leg_handles, fontsize=6.5, frameon=False,
           loc='lower center', ncol=len(CONDITIONS),
           title='Condition (dot colour)', title_fontsize=6,
           bbox_to_anchor=(0.5, -0.06))

fig.tight_layout(rect=[0, 0.06, 1, 1])
fig.savefig(FIG_DIR / 'fig05_occupancy_vs_scfc.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"✓ Occupancy vs SC-FC saved  (rate: {N_rate} subjects | BOLD: {N_bold} subjects)")


--- ## 5.3  Pooled Castro-Style Brain-State Analysis

### Why pooling matters

When centroids are fitted **per subject** (§5.1–§5.2) the k-means solution varies between subjects — state *S0* for subject A and state *S0* for subject B may refer to completely different patterns.  Cross-condition comparison of occupancy is then invalid.

The Castro (2024) method fixes this by:

1. **Pool** all subjects' phase-coherence vectors into a single matrix.
2. Run **one** k-means → shared centroids in a common state space.
3. Rank centroids by SC-FC coupling (Pearson *r* against a reference SC).
4. Assign **every subject** to these shared centroids → comparable occupancy.

We implement two SC-reference variants:

- **Pooled-ref**: centroids correlated against the mean SC across all subjects (condition-agnostic reference — used for sorting and the x-axis).
- **Subject-specific SFC**: same centroids, but *r* is computed against each subject's own SC matrix.  This gives a personalised SFC x-axis for the regression scatter.

> **Expected finding (Castro 2024 / Demertzi 2019):**  
> Conscious controls (CNT) preferentially occupy low-SFC (complex, anti-correlated) states.  DoC patients (COMA/UWS) are trapped in high-SFC (anatomy-constrained) states.  The occupancy–SFC slope should be *negative* for CNT and *positive* (or flat) for COMA/UWS.

In [ ]:
# ══ §5.3a  Pool phase patterns and run one shared k-means ══════════════════
from tvbtoolkit.analysis.brain_states import (
    phase_patterns, cluster_brain_states, sfc_sort_centroids,
    _compute_occupancy, _compute_transition_matrix,
)
from scipy.stats import pearsonr as _pearsonr

def _safe_pearson(a, b):
    """Pearson r between two 1-D vectors; returns NaN on degenerate input."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    mask = np.isfinite(a) & np.isfinite(b)
    if mask.sum() < 3 or np.std(a[mask]) < 1e-12 or np.std(b[mask]) < 1e-12:
        return np.nan
    return float(_pearsonr(a[mask], b[mask])[0])

# ── 1. Extract phase patterns for every available subject ─────────────────
print('Extracting pooled phase patterns (firing rates) ...')

pool_patterns_rate = []   # list[np.ndarray shape (T_i, n_edges)]
pool_meta_rate     = []   # list[dict] cond, subj_idx, sc_vec, n_rows
sc_all_rate        = []   # list[np.ndarray n_regions]

for cond in CONDITIONS:
    for s, subj in enumerate(data[cond]):
        rate = np.asarray(subj['rate'], dtype=float)
        sc   = np.asarray(subj['sc'],   dtype=float)
        try:
            pats, gsync, iu, ju = phase_patterns(
                rate,
                pipeline='firing_rate',
                bandpass_hz=BANDPASS_RATE_HZ,
                filter_order=FILTER_ORDER_RATE,
                dt_ms=DT_MS,
                transient_ms=TRANSIENT_MS,
            )
        except Exception as exc:
            print(f'  Warning [{cond} s{s}] phase_patterns failed: {exc}')
            continue
        if pats.shape[0] < N_STATES:
            print(f'  Skip [{cond} s{s}]: only {pats.shape[0]} time points')
            continue
        n_edges = pats.shape[1]
        n_regions = sc.shape[0]
        sc_vec = sc[np.triu_indices(n_regions, k=1)]
        pool_patterns_rate.append(pats)
        pool_meta_rate.append(dict(
            cond=cond, subj=s, sc_vec=sc_vec, n_rows=pats.shape[0]
        ))
        sc_all_rate.append(sc)

print(f'  Subjects with valid patterns: {len(pool_meta_rate)}')
print(f'  Total phase vectors:          '
      f'{sum(m["n_rows"] for m in pool_meta_rate)}')

# ── 2. Pool into one matrix ───────────────────────────────────────────────
X_rate = np.vstack(pool_patterns_rate).astype(np.float32)   # (N_total, n_edges)
print(f'  Pooled matrix shape: {X_rate.shape}')

# ── 3. Reference SC = mean over all pooled subjects ───────────────────────
sc_ref_mat   = np.mean(np.stack(sc_all_rate, axis=0), axis=0)  # (n_regions, n_regions)
n_reg        = sc_ref_mat.shape[0]
sc_ref_vec   = sc_ref_mat[np.triu_indices(n_reg, k=1)]         # (n_edges,)

# ── 4. Shared k-means ─────────────────────────────────────────────────────
print('Running pooled k-means ...')
labels_pool_rate, centers_pool_rate = cluster_brain_states(
    X_rate,
    n_states=N_STATES,
    random_seed=RANDOM_SEED_KMEANS,
    n_init=N_INIT_KMEANS,
    max_iter=MAX_ITER_KMEANS,
)
print(f'  Cluster sizes: {np.bincount(labels_pool_rate)}')

# ── 5. Sort centroids by SFC against pooled reference SC ─────────────────
sfc_ref_raw = np.array([_safe_pearson(c, sc_ref_vec) for c in centers_pool_rate])
sort_order  = np.argsort(np.nan_to_num(sfc_ref_raw, nan=np.inf))
centers_sorted_rate = centers_pool_rate[sort_order]   # (K, n_edges) sorted low→high SFC
sfc_ref_sorted      = sfc_ref_raw[sort_order]         # SFC of each shared centroid

# Remap labels so label k corresponds to the k-th sorted centroid
label_remap = np.empty_like(sort_order)
label_remap[sort_order] = np.arange(len(sort_order))
labels_pool_rate_sorted = label_remap[labels_pool_rate]

print(f'  SFC of shared centroids (low→high): {np.round(sfc_ref_sorted, 3)}')
print('\nPooled k-means (firing rates): complete.')

In [ ]:
# ══ §5.3b  Pooled phase patterns — BOLD ════════════════════════════════════
print('Extracting pooled phase patterns (BOLD) ...')

pool_patterns_bold = []
pool_meta_bold     = []
sc_all_bold        = []

for cond in CONDITIONS:
    for s, subj in enumerate(data[cond]):
        bold = np.asarray(subj['bold'], dtype=float)
        sc   = np.asarray(subj['sc'],   dtype=float)
        if bold.shape[0] < 30:
            continue
        try:
            pats, gsync, iu_b, ju_b = phase_patterns(
                bold,
                pipeline='brain_act_legacy',
                tr_seconds=TR_SECONDS,
                bandpass_hz=BANDPASS_BOLD_HZ,
                filter_order=FILTER_ORDER_BOLD,
            )
        except Exception as exc:
            print(f'  Warning [{cond} s{s}]: {exc}')
            continue
        if pats.shape[0] < N_STATES:
            continue
        n_regions_b = sc.shape[0]
        sc_vec_b    = sc[np.triu_indices(n_regions_b, k=1)]
        pool_patterns_bold.append(pats)
        pool_meta_bold.append(dict(
            cond=cond, subj=s, sc_vec=sc_vec_b, n_rows=pats.shape[0]
        ))
        sc_all_bold.append(sc)

print(f'  Subjects with valid BOLD patterns: {len(pool_meta_bold)}')
print(f'  Total BOLD time points:            '
      f'{sum(m["n_rows"] for m in pool_meta_bold)}')

if len(pool_patterns_bold) >= N_STATES:
    X_bold          = np.vstack(pool_patterns_bold).astype(np.float32)
    sc_ref_mat_bold = np.mean(np.stack(sc_all_bold, axis=0), axis=0)
    sc_ref_vec_bold = sc_ref_mat_bold[np.triu_indices(sc_ref_mat_bold.shape[0], k=1)]

    print(f'  Pooled BOLD matrix shape: {X_bold.shape}')
    print('Running pooled BOLD k-means ...')
    labels_pool_bold, centers_pool_bold = cluster_brain_states(
        X_bold, n_states=N_STATES,
        random_seed=RANDOM_SEED_KMEANS,
        n_init=N_INIT_KMEANS, max_iter=MAX_ITER_KMEANS,
    )
    sfc_ref_raw_bold = np.array([_safe_pearson(c, sc_ref_vec_bold)
                                  for c in centers_pool_bold])
    sort_order_bold  = np.argsort(np.nan_to_num(sfc_ref_raw_bold, nan=np.inf))
    centers_sorted_bold   = centers_pool_bold[sort_order_bold]
    sfc_ref_sorted_bold   = sfc_ref_raw_bold[sort_order_bold]
    label_remap_bold      = np.empty_like(sort_order_bold)
    label_remap_bold[sort_order_bold] = np.arange(len(sort_order_bold))
    labels_pool_bold_sorted = label_remap_bold[labels_pool_bold]
    print(f'  SFC of shared BOLD centroids (low→high): {np.round(sfc_ref_sorted_bold, 3)}')
else:
    print('  Insufficient BOLD data for pooled analysis — skipping.')
    X_bold = None

print('\nPooled k-means (BOLD): complete.')

In [ ]:
# ══ §5.3c  Per-subject occupancy and subject-specific SFC ══════════════════
# For each subject: slice their global labels, compute occupancy in shared
# state space, and compute SFC of each shared centroid vs their own SC.

def build_pooled_occ_sfc(pool_meta, labels_sorted, centers_sorted,
                          sfc_ref_sorted, n_states=N_STATES):
    """
    Returns
    -------
    records : list[dict]
        One dict per subject containing:
          cond         : str
          subj         : int
          occupancy    : np.ndarray (n_states,)  — shared-centroid occupancy
          sfc_ref      : np.ndarray (n_states,)  — SFC vs pooled-ref SC
          sfc_sub      : np.ndarray (n_states,)  — SFC vs subject's own SC
    """
    records = []
    row_ptr = 0
    for meta in pool_meta:
        a, b = row_ptr, row_ptr + meta['n_rows']
        lab  = labels_sorted[a:b]
        occ  = _compute_occupancy(lab, n_states)
        # Subject-specific SFC of each shared centroid
        sc_vec_sub = meta['sc_vec']
        sfc_sub = np.array([_safe_pearson(c, sc_vec_sub)
                             for c in centers_sorted])
        records.append(dict(
            cond=meta['cond'],
            subj=meta['subj'],
            occupancy=occ,
            sfc_ref=sfc_ref_sorted.copy(),
            sfc_sub=sfc_sub,
        ))
        row_ptr = b
    return records

# Firing rates
pooled_records_rate = build_pooled_occ_sfc(
    pool_meta_rate, labels_pool_rate_sorted,
    centers_sorted_rate, sfc_ref_sorted
)
print(f'Pooled records (firing rates): {len(pooled_records_rate)} subjects')

# BOLD (if computed)
if X_bold is not None:
    pooled_records_bold = build_pooled_occ_sfc(
        pool_meta_bold, labels_pool_bold_sorted,
        centers_sorted_bold, sfc_ref_sorted_bold
    )
    print(f'Pooled records (BOLD):         {len(pooled_records_bold)} subjects')
else:
    pooled_records_bold = []

# Quick sanity: occupancy for first CNT subject
first_cnt = next((r for r in pooled_records_rate if r['cond'] == 'CNT'), None)
if first_cnt is not None:
    print(f'  CNT s0 occupancy: {np.round(first_cnt["occupancy"]*100, 1)} %')
    print(f'  CNT s0 SFC(ref):  {np.round(first_cnt["sfc_ref"], 3)}')
    print(f'  CNT s0 SFC(sub):  {np.round(first_cnt["sfc_sub"], 3)}')
print('\nPer-subject occupancy + SFC: complete.')

In [ ]:
# ══ §5.3d  Castro-style violin plots — occupancy per shared state ══════════
# One violin plot per brain state (K panels), one violin per condition.
# Mirrors Castro et al. 2024 Fig 3 layout.

from scipy.stats import mannwhitneyu as _mwu

def _violin_plot_pooled(
    records, sfc_ref, title_prefix='',
    fig_path=None, n_states=N_STATES
):
    """Castro-style violin plots: K panels, each showing occupancy per condition.

    Parameters
    ----------
    records   : list[dict]  — from build_pooled_occ_sfc()
    sfc_ref   : np.ndarray (n_states,) — SFC of each shared centroid vs ref SC
    """
    if not records:
        print(f'No records for {title_prefix} — skipping.')
        return

    K = n_states
    fig, axes = plt.subplots(1, K, figsize=(2.4 * K, 3.6), sharey=False)
    if K == 1:
        axes = [axes]

    for k, ax in enumerate(axes):
        cond_data = []
        for cond in CONDITIONS:
            vals = np.array([r['occupancy'][k] * 100
                             for r in records if r['cond'] == cond])
            cond_data.append(vals)

        positions = np.arange(len(CONDITIONS))
        vp = ax.violinplot(
            [d if len(d) > 0 else [0.0] for d in cond_data],
            positions=positions,
            showmedians=True,
            showextrema=False,
            widths=0.6,
        )
        # Colour violins by condition
        for body, cond in zip(vp['bodies'], CONDITIONS):
            body.set_facecolor(COND_COLORS.get(cond, '#aaaaaa'))
            body.set_alpha(0.75)
            body.set_edgecolor('black')
            body.set_linewidth(0.6)
        if 'cmedians' in vp:
            vp['cmedians'].set_color('black')
            vp['cmedians'].set_linewidth(1.5)

        # Scatter individual subject points with jitter
        rng_jitter = np.random.default_rng(7)
        for xi, (vals, cond) in enumerate(zip(cond_data, CONDITIONS)):
            if len(vals) == 0:
                continue
            jitter = rng_jitter.uniform(-0.12, 0.12, size=len(vals))
            ax.scatter(xi + jitter, vals,
                       color=COND_COLORS.get(cond, '#aaaaaa'),
                       s=12, alpha=0.85, edgecolors='white',
                       linewidths=0.3, zorder=4)

        # Pairwise significance: CNT vs each DoC group (Mann-Whitney U)
        cnt_idx  = CONDITIONS.index('CNT') if 'CNT' in CONDITIONS else -1
        cnt_vals = cond_data[cnt_idx] if cnt_idx >= 0 else np.array([])
        sig_pairs = []
        for xi, cond in enumerate(CONDITIONS):
            if cond == 'CNT' or len(cond_data[xi]) == 0 or len(cnt_vals) == 0:
                continue
            _, p = _mwu(cnt_vals, cond_data[xi], alternative='two-sided')
            stars = ('***' if p < 0.001 else '**' if p < 0.01
                     else '*' if p < 0.05 else 'ns')
            if stars != 'ns':
                sig_pairs.append((cnt_idx, xi, stars))

        # Draw significance brackets above the violin plot
        ax.autoscale()
        y_top = ax.get_ylim()[1]
        for bi, (xi1, xi2, stars) in enumerate(sig_pairs):
            yb = y_top * (1.06 + 0.10 * bi)
            ax.plot([xi1, xi1, xi2, xi2],
                    [yb * 0.99, yb, yb, yb * 0.99],
                    lw=0.8, color='black')
            ax.text((xi1 + xi2) / 2, yb * 1.005, stars,
                    ha='center', va='bottom', fontsize=6)
        ax.set_ylim(0, ax.get_ylim()[1] * (1.05 + 0.12 * len(sig_pairs)))

        sfc_k = sfc_ref[k] if k < len(sfc_ref) and np.isfinite(sfc_ref[k]) else float('nan')
        ax.set_xticks(positions)
        ax.set_xticklabels(CONDITIONS, fontsize=6, rotation=30, ha='right')
        ax.set_ylabel('Occupancy (%)' if k == 0 else '', fontsize=7)
        ax.set_title(f'State S{k}\nSFC={sfc_k:.3f}', fontsize=7, fontweight='bold')

    fig.suptitle(
        f'{title_prefix}\nPooled shared-centroid occupancy by condition '
        f'(sorted low\u2192high SC-FC)',
        fontsize=8, fontweight='bold'
    )
    fig.tight_layout()
    if fig_path:
        fig.savefig(fig_path, dpi=300, bbox_inches='tight')
        print(f'  Saved: {fig_path.name}')
    plt.show()

# ── Firing rates ─────────────────────────────────────────────────────────
_violin_plot_pooled(
    pooled_records_rate,
    sfc_ref=sfc_ref_sorted,
    title_prefix='A  Pooled Brain States — Firing Rates',
    fig_path=FIG_DIR / 'fig06a_pooled_violin_rate.png'
)

# ── BOLD ─────────────────────────────────────────────────────────────────
if pooled_records_bold:
    _violin_plot_pooled(
        pooled_records_bold,
        sfc_ref=sfc_ref_sorted_bold,
        title_prefix='B  Pooled Brain States — BOLD',
        fig_path=FIG_DIR / 'fig06b_pooled_violin_bold.png'
    )
else:
    print('BOLD violin: skipped (insufficient data).')

In [ ]:
# ══ §5.3e  SFC vs Occupancy regression scatter — pooled + subject-specific ═
# Two variants:
#   (A) x = SFC of shared centroid vs pooled-reference SC  (condition-agnostic)
#   (B) x = SFC of shared centroid vs subject's own SC     (personalised)
# y = subject's occupancy in that shared centroid (%)
# One regression line per condition (not per state) — matches Castro Fig 4.

from scipy import stats as _stats
import matplotlib.patches as _mpatches

def _sfc_occ_scatter(
    records, sfc_mode='ref', ax=None,
    title='', n_states=N_STATES
):
    """
    sfc_mode : 'ref'  → use sfc_ref (pooled-reference SC)
               'sub'  → use sfc_sub (subject-specific SC)
    """
    if not records:
        return
    standalone = ax is None
    if standalone:
        fig, ax = plt.subplots(figsize=(4.5, 3.5))

    # Scatter every (subject × state) point coloured by condition
    for cond in CONDITIONS:
        xs, ys = [], []
        for r in records:
            if r['cond'] != cond:
                continue
            sfc_vals = r['sfc_ref'] if sfc_mode == 'ref' else r['sfc_sub']
            for k in range(n_states):
                if np.isfinite(sfc_vals[k]):
                    xs.append(sfc_vals[k])
                    ys.append(r['occupancy'][k] * 100)
        if not xs:
            continue
        xs, ys = np.array(xs), np.array(ys)
        col = COND_COLORS.get(cond, '#aaaaaa')
        ax.scatter(xs, ys, color=col, s=12, alpha=0.55,
                   edgecolors='white', linewidths=0.3, zorder=3)

        # Regression line per condition
        if len(xs) > 2:
            sl, ic, r_, p_, _ = _stats.linregress(xs, ys)
            xl = np.linspace(xs.min(), xs.max(), 80)
            sig = '***' if p_ < 0.001 else '**' if p_ < 0.01\
                  else '*' if p_ < 0.05 else ''
            ax.plot(xl, sl * xl + ic, color=col, lw=1.8, zorder=4,
                    label=f'{cond}  r={r_:.2f}{sig}')
            # 95% CI band
            n_ = len(xs)
            se_ = (np.sqrt(np.sum((ys - (sl*xs + ic))**2) / (n_-2))
                   * np.sqrt(1/n_ + (xl - xs.mean())**2
                             / np.sum((xs - xs.mean())**2)))
            t_  = _stats.t.ppf(0.975, df=n_-2)
            ax.fill_between(xl, sl*xl+ic - t_*se_,
                            sl*xl+ic + t_*se_, color=col, alpha=0.10, zorder=2)

    xlab = ('SC-FC coupling — pooled-ref SC (Pearson r)'
            if sfc_mode == 'ref'
            else 'SC-FC coupling — subject-specific SC (Pearson r)')
    ax.set_xlabel(xlab, fontsize=7)
    ax.set_ylabel('State occupancy (%)', fontsize=7)
    ax.set_title(title, fontsize=8, fontweight='bold', loc='left')
    ax.set_ylim(bottom=0)
    ax.legend(fontsize=6, frameon=False,
              bbox_to_anchor=(1.01, 1), loc='upper left',
              title='Condition', title_fontsize=6)

    if standalone:
        fig.tight_layout()
        return fig

# ── Build combined 2×2 figure (rate + BOLD) × (ref + sub) ────────────────
n_rows = 2 if pooled_records_bold else 1
fig, axes = plt.subplots(n_rows, 2, figsize=(8.0, 3.6 * n_rows))
axes = np.array(axes).reshape(n_rows, 2)

_sfc_occ_scatter(pooled_records_rate, sfc_mode='ref',
                 ax=axes[0, 0],
                 title='A  Firing rate — pooled-ref SFC')
_sfc_occ_scatter(pooled_records_rate, sfc_mode='sub',
                 ax=axes[0, 1],
                 title='B  Firing rate — subject-specific SFC')

if pooled_records_bold:
    _sfc_occ_scatter(pooled_records_bold, sfc_mode='ref',
                     ax=axes[1, 0],
                     title='C  BOLD — pooled-ref SFC')
    _sfc_occ_scatter(pooled_records_bold, sfc_mode='sub',
                     ax=axes[1, 1],
                     title='D  BOLD — subject-specific SFC')

# Shared condition legend
leg_h = [_mpatches.Patch(color=COND_COLORS[c], alpha=0.85, label=c)
         for c in CONDITIONS if c in COND_COLORS]
fig.legend(handles=leg_h, fontsize=6.5, frameon=False,
           loc='lower center', ncol=len(CONDITIONS),
           title='Condition', title_fontsize=6,
           bbox_to_anchor=(0.5, -0.04))

fig.suptitle(
    'SFC vs Occupancy — Pooled Shared Centroids (Castro-style)',
    fontsize=9, fontweight='bold'
)
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(FIG_DIR / 'fig07_pooled_sfc_vs_occ_scatter.png',
            dpi=300, bbox_inches='tight')
plt.show()
print('✓ Pooled SFC vs occupancy scatter saved.')

---
## 6. Lempel-Ziv Complexity (LZc)

### 6.1 Theory — Algorithmic complexity of brain activity

#### Intuition

Imagine you are trying to compress a text file. A file containing the word *AAAAAAAAAA* compresses to almost nothing because there is only one distinct pattern repeated endlessly. A file containing random characters cannot be compressed at all because every substring is novel.

**Lempel-Ziv (LZ76) complexity** applies this compression intuition to neural signals. A brain with a *rich repertoire of activity patterns* — where each moment of activity is somewhat different from all previous moments — will compress poorly (high LZc). A brain trapped in a single repetitive attractor state will compress well (low LZc).

Empirically and theoretically, **consciousness correlates with high LZc**: wakefulness > NREM sleep > ketamine anaesthesia > propofol anaesthesia > UWS (Casali 2013, Schartner 2015).

#### The LZ76 algorithm

Given a binary sequence $S = s_1 s_2 \ldots s_N$:

1. Initialize a **dictionary** with the empty string.
2. Scan from left to right, extending the current **word** $w$ as long as $w$ was seen in $S[1:i]$ (the prefix seen so far).
3. When $w$ is a new pattern not seen before, add it to the dictionary, increment the complexity counter $c$, and start a new word.
4. Continue until the end of the sequence.

Complexity $C(S) = c$ counts the number of distinct substrings needed to describe $S$. This is related to Kolmogorov complexity — the length of the shortest program that generates $S$.

#### From continuous signals to binary sequences

Neural signals are continuous, not binary. We convert them using the **Hilbert envelope** as a threshold:

1. **Preprocess**: mean-subtract then linear-detrend each channel (mirrors TVBSim).
2. **Compute analytic signal**: $z_i(t) = x_i(t) + j\mathcal{H}[x_i(t)]$
3. **Amplitude envelope**: $A_i(t) = |z_i(t)|$
4. **Binarize**: $b_i(t) = 1$ if $A_i(t) > \bar{A}_i$, else $0$
5. **Flatten**: concatenate all channels in time → 1D binary string
6. **Apply LZ76** and normalize by a shuffled surrogate

#### Normalization

Raw LZc depends on sequence length. Normalization:
$$\text{LZc} = \frac{C(S)}{C(S_{\text{shuffled}})}$$

where $S_{\text{shuffled}}$ is the same sequence with elements randomly permuted. This gives a ratio of 1.0 for maximally random activity and approaching 0 for perfectly repetitive activity.

#### Two preprocessing pipelines

| Domain | Preprocessing | What it measures |
|--------|--------------|------------------|
| **Firing rates** | Bandpass 2–80 Hz, then Hilbert binarize | Complexity of fast neural dynamics |
| **BOLD** | Bandpass 0.01–0.20 Hz, then Hilbert binarize | Complexity of slow infra-slow co-activation |

In [ ]:
from scipy.signal import filtfilt, iirfilter
from scipy.stats import zscore

def _bandpass_filter(x: np.ndarray, dt_ms: float, band_hz: tuple, order: int) -> np.ndarray:
    """Apply zero-phase Butterworth bandpass (helper for LZc preprocessing)."""
    nyq = 0.5 / (dt_ms / 1000.0)
    lo, hi = band_hz[0] / nyq, band_hz[1] / nyq
    lo = min(lo, 0.99)  
    hi = min(hi, 0.999)
    b, a = iirfilter(order, (lo, hi), btype='bandpass', ftype='butter', output='ba')
    try:
        return filtfilt(b, a, x, axis=0)
    except ValueError:
        return filtfilt(b, a, x, axis=0, method='gust')


def compute_lzc_rate(subj_data: dict) -> float:
    """LZc on firing rates: discard transient, z-score, bandpass, then LZc."""
    rate = np.asarray(subj_data['rate'], dtype=float)
    t_ms = np.asarray(subj_data['rate_time'], dtype=float)
    dt = float(t_ms[1] - t_ms[0]) if len(t_ms) > 1 else DT_MS
    
    # Discard transient
    n_trans = int(round(TRANSIENT_MS / dt))
    rate = rate[n_trans:]
    
    # Z-score
    rate_z = zscore(rate, axis=0, ddof=1)
    rate_z = np.nan_to_num(rate_z)
    
    # Bandpass (same as brain-state preprocessing)
    rate_f = _bandpass_filter(rate_z, dt, BANDPASS_RATE_HZ, FILTER_ORDER_RATE)
    
    return lzc_multichannel(rate_f)


def compute_lzc_bold(subj_data: dict) -> float:
    """LZc on BOLD: z-score, bandpass, then LZc."""
    bold = np.asarray(subj_data['bold'], dtype=float)
    if bold.shape[0] < 30:
        return float('nan')
    dt_bold_ms = TR_SECONDS * 1000.0
    
    try:
        bold_f = _bandpass_filter(bold, dt_bold_ms, BANDPASS_BOLD_HZ, FILTER_ORDER_BOLD)
    except Exception:
        return float('nan')
    
    return lzc_multichannel(bold_f)


# ── Compute LZc for all subjects × conditions ─────────────────────────────────
print('Computing LZc (firing rates + BOLD)...')

lzc_rate_vals = {c: [] for c in CONDITIONS}
lzc_bold_vals = {c: [] for c in CONDITIONS}

# Multiple seeds for surrogate variability estimate
N_LZC_REPS = 3

for cond in CONDITIONS:
    for subj in data[cond]:
        # Average over a few surrogate seeds for stability
        rvals = [compute_lzc_rate(subj) for _ in range(N_LZC_REPS)]
        bvals = [compute_lzc_bold(subj) for _ in range(N_LZC_REPS)]
        lzc_rate_vals[cond].append(float(np.mean(rvals)))
        lzc_bold_vals[cond].append(float(np.nanmean(bvals)))
    print(f'  {cond}  LZc rate={np.mean(lzc_rate_vals[cond]):.3f}±{np.std(lzc_rate_vals[cond]):.3f}  '
          f'LZc bold={np.nanmean(lzc_bold_vals[cond]):.3f}±{np.nanstd(lzc_bold_vals[cond]):.3f}')

print('\nLZc computation complete.')

In [ ]:
# ── LZc visualisation ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.suptitle('Lempel-Ziv Complexity (normalized)', fontsize=12, fontweight='bold')

def _lzc_bar_panel(ax, lzc_dict, title, ylabel='LZc (normalized)'):
    positions = np.arange(len(CONDITIONS))
    for xi, cond in enumerate(CONDITIONS):
        vals = [v for v in lzc_dict[cond] if np.isfinite(v)]
        if not vals:
            continue
        mu, se = np.mean(vals), np.std(vals) / max(np.sqrt(len(vals)), 1)
        ax.bar(xi, mu, color=COND_COLORS[cond], alpha=0.8, edgecolor='black', lw=0.8)
        ax.errorbar(xi, mu, yerr=se, fmt='none', color='black', capsize=5, lw=1.5)
        # Individual data points
        jitter = np.random.default_rng(0).uniform(-0.15, 0.15, len(vals))
        ax.scatter(xi + jitter, vals, color='black', s=20, alpha=0.7, zorder=5)
    ax.set_xticks(positions)
    ax.set_xticklabels(CONDITIONS, fontsize=11)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontsize=11)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)
    # Expected ordering annotation
    ax.text(0.02, 0.97, 'Expected: CNT > EMCS > MCS > UWS > COMA', transform=ax.transAxes,
            fontsize=8, va='top', style='italic', color='grey')

_lzc_bar_panel(axes[0], lzc_rate_vals, 'Firing Rates (2–80 Hz filtered)')
_lzc_bar_panel(axes[1], lzc_bold_vals, 'BOLD (0.01–0.20 Hz filtered)')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig03_lzc.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure: LZc distributions stratified by sedation — Nature NN style ─────────
# Each subject is one data point.  Conditions on x-axis; sedated/unsedated shown
# as paired violin + strip within each condition.
# All pairwise Mann-Whitney U tests with Benjamini-Hochberg FDR correction.

from scipy.stats import mannwhitneyu as _mwu
from itertools import combinations as _combs
import matplotlib.patches as _mpatches

try:
    from statsmodels.stats.multitest import multipletests as _mt
    _HAS_STATSMODELS = True
except ImportError:
    _HAS_STATSMODELS = False

def _add_stat_brackets(ax, pairs, pvals, all_vals_flat, x_positions, sig_only=True, fontsize=6):
    """Draw significance brackets above violins."""
    y_top  = ax.get_ylim()[1]
    y_step = (y_top - min(all_vals_flat)) * 0.07
    drawn  = []
    level  = 0
    for (i, j), pv in zip(pairs, pvals):
        stars = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else 'ns'
        if sig_only and stars == 'ns':
            continue
        y = y_top + y_step * (level + 1)
        xi, xj = x_positions[i], x_positions[j]
        ax.plot([xi, xi, xj, xj], [y - y_step*0.3, y, y, y - y_step*0.3],
                lw=0.8, color='#333333')
        ax.text((xi + xj) / 2, y + y_step*0.05, stars,
                ha='center', va='bottom', fontsize=fontsize, color='#333333')
        drawn.append((i, j, stars))
        level += 1
    # Expand y-limit if brackets were added
    if drawn:
        ax.set_ylim(ax.get_ylim()[0], y_top + y_step * (level + 1.5))
    return drawn

# ── Collect LZc values with sedation label ────────────────────────────────────
lzc_by_sedation = {"Sedated": [], "Unsedated": []}
lzc_by_cond_sed = {c: {"Sedated": [], "Unsedated": []} for c in CONDITIONS}

for cond in CONDITIONS:
    sed_label = SEDATION_MAP.get(cond, "Unsedated")
    vals = [v for v in lzc_rate_vals.get(cond, []) if np.isfinite(v)]
    lzc_by_sedation[sed_label].extend(vals)
    lzc_by_cond_sed[cond][sed_label].extend(vals)

# ── Pairwise stats across ALL conditions ─────────────────────────────────────
all_cond_vals  = [np.array([v for v in lzc_rate_vals.get(c, []) if np.isfinite(v)]) for c in CONDITIONS]
pairs_idx      = [(i, j) for i, j in _combs(range(len(CONDITIONS)), 2)
                  if len(all_cond_vals[i]) > 1 and len(all_cond_vals[j]) > 1]
pvals_raw = []
for i, j in pairs_idx:
    _, pv = _mwu(all_cond_vals[i], all_cond_vals[j], alternative='two-sided')
    pvals_raw.append(pv)

# FDR correction
if _HAS_STATSMODELS and pvals_raw:
    _, pvals_corr, _, _ = _mt(pvals_raw, alpha=0.05, method='fdr_bh')
else:
    pvals_corr = pvals_raw
    print("  Note: statsmodels not available — p-values not FDR-corrected")

# ── Figure layout: 2 panels ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(7.08, 3.4))

# ─── Panel A: LZc per condition (firing rates) ───────────────────────────────
ax = axes[0]
ax.set_title('A  LZc (firing rates) by condition', fontsize=8, loc='left', fontweight='bold')
rng_jit = np.random.default_rng(7)
x_pos   = list(range(len(CONDITIONS)))
all_vals_flat = [v for c in CONDITIONS for v in lzc_rate_vals.get(c, []) if np.isfinite(v)]

for xi, cond in enumerate(CONDITIONS):
    vals = np.array([v for v in lzc_rate_vals.get(cond, []) if np.isfinite(v)])
    if len(vals) == 0:
        continue
    col = COND_COLORS[cond]

    # Violin
    if len(vals) > 2:
        vp = ax.violinplot([vals], positions=[xi], widths=0.55,
                           showmedians=False, showextrema=False)
        for body in vp['bodies']:
            body.set_facecolor(col)
            body.set_edgecolor(col)
            body.set_alpha(0.35)

    # Strip (individual subjects)
    jitter = rng_jit.uniform(-0.1, 0.1, len(vals))
    ax.scatter(xi + jitter, vals, color=col, s=22,
               alpha=0.9, edgecolors='white', linewidths=0.4, zorder=5)

    # Median line
    ax.hlines(np.median(vals), xi - 0.23, xi + 0.23,
              lw=2.0, color=col, zorder=6)

ax.set_xticks(x_pos)
ax.set_xticklabels(CONDITIONS, fontsize=8)
ax.set_ylabel('LZc (normalized)', fontsize=8)

# Add brackets
drawn = _add_stat_brackets(ax, pairs_idx, pvals_corr, all_vals_flat, x_pos, sig_only=True)
if drawn:
    ax.text(0.02, 0.02,
            'FDR-corrected Mann-Whitney U\n* p<0.05  ** p<0.01  *** p<0.001',
            transform=ax.transAxes, fontsize=5.5, va='bottom', color='#666666')

# ─── Panel B: LZc stratified by sedation ─────────────────────────────────────
ax2 = axes[1]
ax2.set_title('B  LZc by sedation status', fontsize=8, loc='left', fontweight='bold')

sed_labels   = ["Unsedated", "Sedated"]
vals_sed_all = [np.array([v for v in lzc_by_sedation[s] if np.isfinite(v)]) for s in sed_labels]
x_pos_sed    = list(range(len(sed_labels)))

for xi, (sed, vals) in enumerate(zip(sed_labels, vals_sed_all)):
    col = SEDATION_COLORS[sed]
    if len(vals) == 0:
        continue
    if len(vals) > 2:
        vp = ax2.violinplot([vals], positions=[xi], widths=0.55,
                            showmedians=False, showextrema=False)
        for body in vp['bodies']:
            body.set_facecolor(col)
            body.set_edgecolor(col)
            body.set_alpha(0.35)
    jitter = rng_jit.uniform(-0.12, 0.12, len(vals))
    # Colour each dot by its condition
    for cond in CONDITIONS:
        if SEDATION_MAP.get(cond) != sed:
            continue
        cond_vals = np.array([v for v in lzc_by_cond_sed[cond][sed] if np.isfinite(v)])
        if len(cond_vals) == 0:
            continue
        cj = rng_jit.uniform(-0.12, 0.12, len(cond_vals))
        ax2.scatter(xi + cj, cond_vals, color=COND_COLORS[cond], s=22,
                    alpha=0.9, edgecolors='white', linewidths=0.4, zorder=5,
                    label=cond)
    ax2.hlines(np.median(vals), xi - 0.23, xi + 0.23,
               lw=2.0, color=col, zorder=6)
    ax2.text(xi, ax2.get_ylim()[0] - 0.01, f'n={len(vals)}',
             ha='center', va='top', fontsize=6.5, color='#555555')

# Sedation pairwise stat
if len(vals_sed_all[0]) > 1 and len(vals_sed_all[1]) > 1:
    _, pv_sed = _mwu(vals_sed_all[0], vals_sed_all[1], alternative='two-sided')
    stars_sed = '***' if pv_sed < 0.001 else '**' if pv_sed < 0.01 else '*' if pv_sed < 0.05 else 'ns'
    y_b = max(np.max(vals_sed_all[0]), np.max(vals_sed_all[1])) * 1.06
    ax2.plot([0, 0, 1, 1], [y_b, y_b*1.03, y_b*1.03, y_b], lw=0.8, color='#333333')
    ax2.text(0.5, y_b*1.04, stars_sed, ha='center', va='bottom', fontsize=7, color='#333333')
    ax2.set_ylim(ax2.get_ylim()[0], y_b * 1.15)
    print(f"  Sedated vs Unsedated LZc:  p = {pv_sed:.4f}  ({stars_sed})")

ax2.set_xticks(x_pos_sed)
ax2.set_xticklabels(sed_labels, fontsize=8)
ax2.set_ylabel('LZc (normalized)', fontsize=8)

# Condition legend (coloured dots)
handles_leg = [_mpatches.Patch(color=COND_COLORS[c], label=c, alpha=0.85) for c in CONDITIONS if c in CONDITIONS]
ax2.legend(handles=handles_leg, fontsize=6, frameon=False, title='Condition',
           title_fontsize=6, loc='upper right')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig06_lzc_sedation.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ LZc sedation figure saved")
if drawn:
    print(f"  Significant pairwise comparisons (FDR-corrected): {len(drawn)}")


---
## 7. Perturbational Complexity Index (PCI)

### 7.1 Why LZc alone is not enough — the perturbation requirement

Spontaneous LZc has a fundamental problem: **high complexity does not always mean high consciousness**. A desynchronized brain in UWS can produce high LZc purely from the noise of disconnected random activity — there is no organized dynamics to make patterns repetitive.

Casali et al. (2013) solved this by measuring **causal** complexity: they asked not "how complex is ongoing activity?" but rather "how rich is the brain's *response* to a perturbation?"

The insight: a truly conscious brain (CNT) receives a TMS pulse and responds with a **cascade of spatiotemporally complex, propagating signals** — the perturbation spreads across the network in a rich, heterogeneous pattern. An unconscious brain (UWS) shows either **silence** (global inhibition) or a **simple uniform echo** that dies quickly.

### 7.2 The Casali PCI algorithm (step by step)

#### Preparation: Multiple trials with the same stimulus

PCI is computed from $N_\text{trials}$ separate trials. In TMS-EEG, these are separate TMS pulses. In simulation (TVBSim approach), these are separate simulation runs with different random seeds but the same external-input perturbation applied at the same time $t_\text{stim}$.

**TVBSim defaults** (replicated exactly in TVBToolkit):
- $N_\text{trials} = 100$ (we use 100; TVBSim reference uses 5)
- $t_\text{analysis} = 300$ ms (window before and after stimulus)
- $N_\text{shuffles} = 10$ (surrogate iterations for threshold)
- Percentile = 100%

#### Step 1: Extract stimulus window from each trial

For trial $k$, extract the signal in a window $[-t_\text{analysis}, +t_\text{analysis}]$ around $t_\text{stim}$:
$$\mathbf{X}^{(k)} = \text{signal}[:, t_\text{stim} - N_\text{bins} : t_\text{stim} + N_\text{bins}]$$
Shape: $(N_\text{sources}, 2N_\text{bins})$ where $N_\text{bins} = t_\text{analysis} / \Delta t$.

Stack: $\mathbf{X} = [\mathbf{X}^{(1)}, \ldots, \mathbf{X}^{(N_\text{trials})}]$ with shape $(N_\text{trials}, N_\text{sources}, 2N_\text{bins})$.

#### Step 2: Baseline normalization

For each trial and source, normalize relative to pre-stimulus baseline ($t < t_\text{stim}$):
$$\tilde{x}_{i}^{(k)}(t) = \frac{x_i^{(k)}(t) / \mu_i^{(k)} - 1}{\sigma_i^{(k)}}$$
where $\mu_i^{(k)}$ and $\sigma_i^{(k)}$ are the pre-stimulus mean and std of the fractional change.

#### Step 3: Bootstrap threshold via surrogate shuffling

To determine what counts as a "significant" post-stimulus response:
1. Shuffle the pre-stimulus data in time ($N_\text{shuffles} = 10$ times)
2. Average across trials (this is the null signal that would occur by chance)
3. Find the maximum absolute value across all surrogates → this is the threshold $\theta$

$$\theta = \max_k \left| \bar{\tilde{x}}^{\text{shuffle}, k} \right|$$

#### Step 4: Binarize

$$b_i^{(k)}(t) = \begin{cases} 1 & \text{if } \tilde{x}_i^{(k)}(t) > \theta \\ 0 & \text{otherwise} \end{cases}$$

#### Step 5: Sort channels and compute LZc

Sort channels by total activity (descending) — this is the Casali sorting:
$$\mathbf{b}^{(k)}_\text{sorted} = \text{argsort}\left(\sum_t b_i^{(k)}(t)\right)_{\text{desc}} $$

Apply the **2D Lempel-Ziv complexity** (scanning column-by-column through the binary matrix):
$$\text{LZ}_{2D}(\mathbf{b}^{(k)}_\text{sorted}) = C$$

#### Step 6: Normalize

$$\text{PCI}^{(k)} = \frac{\text{LZ}_{2D}(\mathbf{b}^{(k)}_\text{sorted})}{S}$$
where $S = \frac{L \cdot H}{\log_2 L}$, $L = N_\text{sources} \times N_\text{bins,post}$, and $H$ is the binary Shannon entropy of $\mathbf{b}$.

#### Final PCI: mean across trials
$$\text{PCI} = \frac{1}{N_\text{trials}} \sum_k \text{PCI}^{(k)}$$

### 7.3 Why BOLD cannot be used for PCI

The Casali algorithm operates on the **immediate response window** around the stimulus (±300 ms). The haemodynamic response function (HRF) peaks at approximately 6,000 ms — twenty times the analysis window. BOLD simply cannot follow a TMS response; the signal remains flat for several seconds before any haemodynamic effect would begin. PCI from BOLD would measure nothing but baseline noise.

In [ ]:
# ── Multi-trial PCI computation ───────────────────────────────────────────────
#
# Priority:
#   1. Load real PCI trial NPZ files from SIM_PCI_DIR  (preferred)
#      Written by brain_act_simulations_v2.ipynb §6 (N_TRIALS_PCI = 100 per subject)
#   2. Fall back to synthetic stimulation-response windows if files not found
#
# Each trial NPZ contains:
#   time_ms          (T,)         — post-transient time axis
#   rate             (T, N_reg)   — firing rates
#   stim_onset_ms    (1,)         — pseudo-stimulus onset in total-sim coordinates
#   t_analysis_ms    (1,)         — one-sided analysis window (300 ms)
#   trial_seed, noise_alpha
#
# The analysis:
#   ┌─── T_analysis ───┬─── T_analysis ───┐
#   │   pre-stim       │   post-stim      │
#   └──────────────────┴──────────────────┘
#              ↑ stim_onset_ms
# ─────────────────────────────────────────────────────────────────────────────

def _load_pci_trials(
    subject_entry: dict,
    pci_dir: Path,
    n_trials: int,
    t_analysis_ms: float,
) -> "tuple[list[np.ndarray] | None, int | None]":
    """Load real PCI trial NPZ files and return pre-cut windows.

    Returns (trial_windows, nbins) where each element of trial_windows is
    (n_regions, 2*nbins) — the format expected by pci_casali_like_multi_trial.
    Returns (None, None) if files are missing or the window cannot be cut.
    """
    scenario_key = subject_entry.get("scenario_key", "private_alpha0")
    cohort       = subject_entry.get("cohort", "")
    subject_id   = subject_entry.get("subject_id", "")

    trial_dir  = pci_dir / scenario_key / cohort / subject_id
    if not trial_dir.exists():
        return None, None

    trial_files = sorted(trial_dir.glob("trial_*.npz"))[:n_trials]
    if len(trial_files) < n_trials:
        return None, None   # not enough trials yet

    trial_windows = []
    nbins_ref = None

    for trial_path in trial_files:
        try:
            d = np.load(trial_path.resolve() if trial_path.is_symlink() else trial_path,
                        allow_pickle=True)
        except Exception as exc:
            print(f"  Could not load {trial_path}: {exc}")
            return None, None

        x         = np.asarray(d["rate"],          dtype=float)   # (T, N_reg)
        t_ms      = np.asarray(d["time_ms"],        dtype=float)   # (T,)
        stim_ms   = float(d["stim_onset_ms"][0])
        t_ana     = float(d["t_analysis_ms"][0])

        # Compute sample-level quantities
        dt_ms  = float(np.median(np.diff(t_ms))) if len(t_ms) > 1 else DT_MS
        nbins  = int(round(t_ana / dt_ms))   # one-sided bins

        if nbins_ref is None:
            nbins_ref = nbins
        elif nbins != nbins_ref:
            # Mismatched windows — cannot stack safely
            return None, None

        # time_ms[0] = start of post-transient region
        stim_idx = int(round((stim_ms - t_ms[0]) / dt_ms))
        pre_start = stim_idx - nbins
        post_end  = stim_idx + nbins
        if pre_start < 0 or post_end > x.shape[0]:
            print(f"  Window out of bounds for {trial_path.name}: "
                  f"stim_idx={stim_idx} nbins={nbins} n_samples={x.shape[0]}")
            return None, None

        window = x[pre_start:post_end, :]   # (2*nbins, N_reg)
        trial_windows.append(window.T)      # (N_reg, 2*nbins)

    return trial_windows, nbins_ref


def _generate_pci_trials_synthetic(
    subject_data: dict,
    n_trials: int = N_TRIALS_PCI,
    t_analysis_ms: float = T_ANALYSIS_MS_PCI,
) -> "tuple[list[np.ndarray], int]":
    """Synthetic fallback: generate stimulation trials from spontaneous firing rates.

    Adds a condition-scaled stimulus response to the post-stimulus half.
    Used only when real PCI trial NPZ files are not available.

    Returns (trial_windows, nbins_mid) where each trial is (n_regions, 2*nbins).
    """
    cond       = subject_data["condition"]
    rate       = np.asarray(subject_data["rate"])   # (n_bins, n_regions)
    t_rate     = np.asarray(subject_data["rate_time"])
    n_bins, n_regions = rate.shape
    dt_ms      = float(t_rate[1] - t_rate[0]) if len(t_rate) > 1 else DT_MS

    nbins = int(round(t_analysis_ms / dt_ms))
    stim_idx = max(nbins, min(n_bins // 2, n_bins - nbins))

    resp_amp       = {"CNT": 6.0, "EMCS": 4.0, "MCS": 2.5, "UWS": 0.6, "COMA": 0.3}.get(cond, 1.0)
    spatial_heterog = {"CNT": 0.8, "EMCS": 0.6, "MCS": 0.4, "UWS": 0.1, "COMA": 0.05}.get(cond, 0.3)
    f_resp         = {"CNT": 35.0, "EMCS": 25.0, "MCS": 15.0, "UWS": 5.0, "COMA": 3.0}.get(cond, 10.0)

    rng = np.random.default_rng(seed=12345)
    trials = []
    for k in range(n_trials):
        offset  = k * (nbins // max(n_trials, 1))
        stim_k  = max(nbins, min(stim_idx + offset, n_bins - nbins))
        window  = rate[stim_k - nbins : stim_k + nbins, :].copy()

        t_post          = np.arange(nbins) * dt_ms
        response_tmpl   = np.exp(-t_post / 50.0) * np.sin(2 * np.pi * f_resp * t_post / 1000.0)
        spatial_w       = np.abs(spatial_heterog * rng.standard_normal(n_regions)
                                 + (1 - spatial_heterog))
        window[nbins:, :] += resp_amp * response_tmpl[:, None] * spatial_w[None, :]
        trials.append(window.T)   # (n_regions, 2*nbins)

    return trials, nbins


# ── Main PCI loop ─────────────────────────────────────────────────────────────
print("Computing multi-trial PCI …")
print(f"  n_trials={N_TRIALS_PCI}  t_analysis={T_ANALYSIS_MS_PCI} ms  "
      f"nshuffles={NSHUFFLES_PCI}  percentile={PERCENTILE_PCI}")

pci_vals         = {c: [] for c in CONDITIONS}
pci_per_trial_all = {c: [] for c in CONDITIONS}

n_real = 0
n_synthetic = 0

for cond in CONDITIONS:
    for subj in data[cond]:
        # ── Try loading real PCI trials first ─────────────────────────────
        trial_windows, nbins_mid = _load_pci_trials(
            subj, SIM_PCI_DIR, N_TRIALS_PCI, T_ANALYSIS_MS_PCI
        )
        used_real = trial_windows is not None

        if not used_real:
            # Fall back to synthetic generation
            trial_windows, nbins_mid = _generate_pci_trials_synthetic(
                subj, n_trials=N_TRIALS_PCI, t_analysis_ms=T_ANALYSIS_MS_PCI
            )

        if used_real:
            n_real += 1
        else:
            n_synthetic += 1

        try:
            mean_pci, pci_trial = pci_casali_like_multi_trial(
                trial_windows,
                stimulation_index=nbins_mid,
                t_analysis_ms=T_ANALYSIS_MS_PCI,
                dt_ms=DT_MS,
                nshuffles=NSHUFFLES_PCI,
                percentile=PERCENTILE_PCI,
            )
        except Exception as exc:
            print(f"  Warning [{cond}]: PCI failed — {exc}")
            mean_pci  = float("nan")
            pci_trial = np.full(N_TRIALS_PCI, float("nan"))

        pci_vals[cond].append(mean_pci)
        pci_per_trial_all[cond].append(pci_trial)

    vals_finite = [v for v in pci_vals[cond] if np.isfinite(v)]
    print(f"  {cond:5s}: mean PCI = {np.mean(vals_finite):.4f} ± {np.std(vals_finite):.4f} "
          f"(n={len(vals_finite)})")

print(f"\nPCI complete.  Real trial files used: {n_real}  |  Synthetic fallback: {n_synthetic}")
if n_synthetic > 0 and n_real == 0:
    print("  ► All PCI estimates used synthetic trials.")
    print("    Run brain_act_simulations_v2.ipynb §6 to generate real PCI trial files.")
elif n_synthetic > 0:
    print(f"  ► {n_synthetic} subject(s) fell back to synthetic trials — check sims_pci/ coverage.")


In [ ]:
# ── PCI visualisation — one data point per subject (mean over 100 trials) ────
# pci_vals[cond] already contains ONE value per subject (mean over N_TRIALS_PCI=100).
# This is the correct compression: pci_casali_like_multi_trial pools all 100 trial
# windows for baseline thresholding, then returns the mean across trial-level PCI scores.
# We do NOT plot 100 per-trial dots — that would misrepresent the measure.
import matplotlib.patches as mpatches
from scipy import stats as _stats

fig, axes = plt.subplots(1, 2, figsize=(7.08, 3.2))
fig.suptitle(
    f'Perturbational Complexity Index — Casali multi-trial method\n'
    f'n_trials = {N_TRIALS_PCI}  |  t_analysis = {T_ANALYSIS_MS_PCI:.0f} ms  |  n_shuffles = {NSHUFFLES_PCI}',
    fontsize=8, fontweight='bold', y=1.01
)

# ── Left: per-subject strip + violin + median bar ─────────────────────────────
ax = axes[0]
rng_jit = np.random.default_rng(42)

_cond_positions = {c: xi for xi, c in enumerate(CONDITIONS)}
_all_vals, _all_pos = [], []

for xi, cond in enumerate(CONDITIONS):
    vals = np.array([v for v in pci_vals[cond] if np.isfinite(v)])
    if len(vals) == 0:
        continue
    jitter = rng_jit.uniform(-0.15, 0.15, len(vals))
    ax.scatter(xi + jitter, vals,
               color=COND_COLORS[cond], s=28, alpha=0.85,
               edgecolors='white', linewidths=0.4, zorder=4)
    # Median bar
    ax.hlines(np.median(vals), xi - 0.28, xi + 0.28,
              lw=2.0, color=COND_COLORS[cond], zorder=5)
    # IQR box
    q25, q75 = np.percentile(vals, [25, 75])
    ax.add_patch(plt.Rectangle(
        (xi - 0.22, q25), 0.44, q75 - q25,
        facecolor=COND_COLORS[cond], alpha=0.22,
        edgecolor=COND_COLORS[cond], linewidth=0.8, zorder=3
    ))
    _all_vals.extend(vals.tolist())
    _all_pos.extend([xi] * len(vals))

ax.set_xticks(range(len(CONDITIONS)))
ax.set_xticklabels(CONDITIONS, fontsize=8)
ax.set_ylabel('PCI (Casali, mean over trials)', fontsize=8)
ax.set_title('One data point per subject', fontsize=8)
ax.text(0.03, 0.97, 'Expected: CNT > EMCS > MCS > UWS > COMA',
        transform=ax.transAxes, fontsize=6.5, va='top', style='italic',
        color='#555555')

# ── Right: condition means ± SD ──────────────────────────────────────────────
ax2 = axes[1]
for xi, cond in enumerate(CONDITIONS):
    vals = np.array([v for v in pci_vals[cond] if np.isfinite(v)])
    if len(vals) == 0:
        continue
    mu, sd = np.mean(vals), np.std(vals)
    ax2.bar(xi, mu, color=COND_COLORS[cond], alpha=0.85,
            edgecolor='white', linewidth=0.6, width=0.6)
    ax2.errorbar(xi, mu, yerr=sd, fmt='none',
                 color='#333333', capsize=3.5, lw=1.2, capthick=1.2)
    ax2.text(xi, -0.003, f'n={len(vals)}',
             ha='center', va='top', fontsize=6, color='#555555')

ax2.set_xticks(range(len(CONDITIONS)))
ax2.set_xticklabels(CONDITIONS, fontsize=8)
ax2.set_ylabel('PCI (mean ± SD)', fontsize=8)
ax2.set_title('Group means', fontsize=8)

fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(FIG_DIR / 'fig04_pci_persubject.png', dpi=300, bbox_inches='tight')
plt.show()
print("✓ PCI figure saved (one data point per subject = mean over 100 trials)")
print(f"\nPCI summary:")
for cond in CONDITIONS:
    vals = [v for v in pci_vals[cond] if np.isfinite(v)]
    if vals:
        print(f"  {cond:5s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}  (n={len(vals)} subjects)")


### 7.4  PCI trial files — connection to `brain_act_simulations_v2.ipynb`

The PCI trial NPZ files are generated by **§6 of `brain_act_simulations_v2.ipynb`**
via `run_pci_trial_job`.  Each trial is a **2 600 ms simulation with a genuine
TMS-like pulse** injected via TVB's `StimuliRegion + PulseTrain` API:

| Parameter | Default | Meaning |
|-----------|---------|---------|
| `STIM_AMPLITUDE_PCI` | 0.00015 kHz (= 0.15 Hz) | Reduced for T=5ms: 8× faster dynamics vs Hugo T=40ms |
| `STIM_DURATION_MS_PCI` | 10 ms | Scaled for T=5ms (50ms × 5/40 ≈ 10ms) |
| `STIM_REGION_PCI` | [18] (`Supp_Motor_Area_L`) | Left premotor BA6 — Casali 2013 primary TMS site |
| `stim_onset_ms` | randomised per trial in [4300, 7700) ms | Ensures full ±300 ms window fits within 8 s trial |

The `_load_pci_trials` loader above reads `stim_onset_ms` from each NPZ and
extracts the $[-300, +300]$ ms window automatically.

**Pre-stim (−300 → 0 ms):** spontaneous baseline pooled across all 10 trials →
surrogate threshold for `binarise_signals`.

**Post-stim (0 → +300 ms):** genuine perturbational response → 2D LZc →
normalised PCI per trial → mean across trials returned.

This matches the TVBSim `parallelized_PCI` / `_calculate_PCI_seed_subset`
pipeline exactly, with `n_trials=10`, `t_analysis=300`, `nshuffles=10`,
`percentile=100.0`.

In [ ]:
# ── Figure: LZc and PCI — combined global comparison (Nature NN style) ─────────
# Panel A: LZc by condition (violin + strip, Wes Anderson palette, pairwise stats)
# Panel B: PCI by condition (same style, matched axes for direct comparison)
# One data point per subject throughout.

from scipy.stats import mannwhitneyu as _mwu3
from itertools import combinations as _combs3
import matplotlib.patches as _mpatches3

try:
    from statsmodels.stats.multitest import multipletests as _mt3
    _HAS_SM3 = True
except ImportError:
    _HAS_SM3 = False

def _stat_brackets(ax, cond_vals, CONDITIONS, y_anchor, y_step, fontsize=6):
    """Add FDR-corrected MWU brackets for significant pairs."""
    n = len(CONDITIONS)
    pairs = [(i, j) for i in range(n) for j in range(i+1, n)
             if len(cond_vals[i]) > 1 and len(cond_vals[j]) > 1]
    if not pairs:
        return
    praw = []
    for i, j in pairs:
        _, pv = _mwu3(cond_vals[i], cond_vals[j], alternative='two-sided')
        praw.append(pv)
    if _HAS_SM3 and praw:
        _, pcorr, _, _ = _mt3(praw, alpha=0.05, method='fdr_bh')
    else:
        pcorr = praw
    level = 0
    for (i, j), pv in zip(pairs, pcorr):
        stars = '***' if pv < 0.001 else '**' if pv < 0.01 else '*' if pv < 0.05 else ''
        if not stars:
            continue
        y = y_anchor + y_step * (level + 1)
        ax.plot([i, i, j, j], [y - y_step*0.3, y, y, y - y_step*0.3],
                lw=0.8, color='#333333')
        ax.text((i + j) / 2, y + y_step*0.05, stars,
                ha='center', va='bottom', fontsize=fontsize, color='#333333')
        level += 1
    if level:
        ax.set_ylim(ax.get_ylim()[0], y_anchor + y_step * (level + 2))

def _violin_strip_panel(ax, vals_dict, CONDITIONS, label, panel_letter):
    """Draw violin + strip plot for one metric."""
    rng = np.random.default_rng(17)
    cond_data = [np.array([v for v in vals_dict.get(c, []) if np.isfinite(v)])
                 for c in CONDITIONS]
    all_flat = np.concatenate([d for d in cond_data if len(d) > 0])

    for xi, (cond, vals) in enumerate(zip(CONDITIONS, cond_data)):
        if len(vals) == 0:
            continue
        col = COND_COLORS[cond]
        # Violin
        if len(vals) > 2:
            vp = ax.violinplot([vals], positions=[xi], widths=0.60,
                               showmedians=False, showextrema=False)
            for body in vp['bodies']:
                body.set_facecolor(col)
                body.set_edgecolor(col)
                body.set_alpha(0.30)
        # IQR box
        q25, q75 = np.percentile(vals, [25, 75])
        ax.add_patch(plt.Rectangle(
            (xi - 0.18, q25), 0.36, q75 - q25,
            facecolor=col, alpha=0.25,
            edgecolor=col, linewidth=0.8, zorder=3
        ))
        # Median line
        ax.hlines(np.median(vals), xi - 0.22, xi + 0.22,
                  lw=2.2, color=col, zorder=5)
        # Individual subjects
        jitter = rng.uniform(-0.12, 0.12, len(vals))
        ax.scatter(xi + jitter, vals, color=col, s=24,
                   alpha=0.90, edgecolors='white', linewidths=0.35, zorder=6)
        # n label
        ax.text(xi, ax.get_ylim()[0] * 1.01 if ax.get_ylim()[0] > 0 else -0.002,
                f'n={len(vals)}', ha='center', va='top', fontsize=5.5, color='#555555')

    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS, fontsize=8)
    ax.set_ylabel(label, fontsize=8)
    ax.set_title(f'{panel_letter}', fontsize=8, loc='left', fontweight='bold')
    ax.text(0.03, 0.97, 'Expected: CNT > EMCS > MCS > UWS > COMA',
            transform=ax.transAxes, fontsize=6, va='top', style='italic', color='#777777')

    # Stat brackets
    if len(all_flat) > 0:
        y_top = np.max(all_flat) * 1.02
        y_step = (np.max(all_flat) - np.min(all_flat)) * 0.06
        _stat_brackets(ax, cond_data, CONDITIONS, y_top, y_step)

    return cond_data

# ── Main figure ───────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(7.08, 3.5))

_violin_strip_panel(axes[0], lzc_rate_vals, CONDITIONS,
                    'LZc — firing rates (normalized)', 'A  Lempel-Ziv Complexity')
_violin_strip_panel(axes[1], pci_vals, CONDITIONS,
                    'PCI — Casali multi-trial (mean 100 trials)', 'B  Perturbational Complexity Index')

# Shared figure annotation
fig.text(0.5, -0.04,
         'FDR-corrected Mann–Whitney U  |  * p<0.05  ** p<0.01  *** p<0.001\n'
         'Each dot = one subject (1 LZc scalar; 1 PCI scalar = mean over 100 trials)',
         ha='center', fontsize=6, color='#555555')

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig07_lzc_pci_combined.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Combined LZc + PCI figure saved")
for cond in CONDITIONS:
    lv = [v for v in lzc_rate_vals.get(cond, []) if np.isfinite(v)]
    pv = [v for v in pci_vals.get(cond, []) if np.isfinite(v)]
    print(f"  {cond:5s}  LZc={np.mean(lv):.3f}±{np.std(lv):.3f} (n={len(lv)})  "
          f"PCI={np.mean(pv):.4f}±{np.std(pv):.4f} (n={len(pv)})")


---
## 8. Summary — Comparing All Three Measures

### What each measure captures

| Measure | Domain | Timescale | What it captures | Expected DoC ordering |
|---------|--------|-----------|------------------|-----------------------|
| **Brain States** (occupancy of high-SC state) | Rates / BOLD | 5 ms / 2.4 s | How much time the brain spends in its most SC-aligned synchrony pattern | CNT < MCS < UWS (UWS stays trapped in SC-driven state) |
| **LZc** | Rates / BOLD | Fast / Slow | Repertoire size of spatiotemporal patterns | CNT > MCS > UWS |
| **PCI** | Rates (firing) | 5 ms (post-stim 300 ms) | Richness of *causal* response to perturbation | CNT > MCS > UWS |

### Why they can disagree

- Spontaneous LZc can be **higher in UWS** if the low-coupling regime produces desynchronized noise that is algorithmically complex.
- Brain states on BOLD may show **no difference** between conditions if the BOLD signal is too short or the frequency band is not well-matched.
- **PCI is the most reliable** discriminator because it is robust to ongoing activity: only the perturbation-driven component above the baseline threshold contributes to PCI.

### The two preprocessing pipelines

**Firing-rate pipeline** (new, `pipeline='firing_rate'`):
```
raw E(t) [Hz] → drop 500 ms transient → z-score → Butterworth 2–80 Hz → Hilbert → phase/envelope
```

**BOLD pipeline** (`pipeline='brain_act_legacy'`):
```
BOLD(t) → z-score (ddof=0) → demean across ROIs → Butterworth 0.01–0.20 Hz → Hilbert + unwrap → phase
```

These are fundamentally different because the HRF blurs neural dynamics by ≈6,000 ms. The two pipelines answer different scientific questions about the same simulation.

In [ ]:
# ── Summary comparison figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 5))
fig.suptitle('All complexity measures — mean ± SE across subjects', 
             fontsize=12, fontweight='bold')

# Helper: occupancy of the highest-SFC state (state K-1 after sorting)
def _last_state_occupancy(bs_list):
    vals = []
    for s in bs_list:
        if s is None or s.occupancy.size == 0:
            continue
        vals.append(float(s.occupancy[-1]))
    return vals

panels = [
    ('Brain States\nHigh-SFC occ. (rates)', {c: _last_state_occupancy(bs_rate[c]) for c in CONDITIONS}),
    ('LZc\nFiring rates', lzc_rate_vals),
    ('LZc\nBOLD', lzc_bold_vals),
    ('PCI\n(multi-trial Casali)', pci_vals),
]

for ax, (title, vals_dict) in zip(axes, panels):
    for xi, cond in enumerate(CONDITIONS):
        vals = [v for v in vals_dict[cond] if np.isfinite(v)]
        if not vals:
            continue
        mu = np.mean(vals)
        se = np.std(vals) / max(np.sqrt(len(vals)), 1)
        ax.bar(xi, mu, color=COND_COLORS[cond], alpha=0.82, edgecolor='black', lw=0.8)
        ax.errorbar(xi, mu, yerr=se, fmt='none', color='black', capsize=5, lw=1.5)
        jitter = np.random.default_rng(42).uniform(-0.12, 0.12, len(vals))
        ax.scatter(xi + jitter, vals, color='black', s=18, alpha=0.65, zorder=5)
    ax.set_xticks(range(len(CONDITIONS)))
    ax.set_xticklabels(CONDITIONS, fontsize=11)
    ax.set_title(title, fontsize=10)
    ax.set_ylim(bottom=0)

fig.tight_layout()
fig.savefig(FIG_DIR / 'fig05_summary.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary table
print('\nSummary table (mean ± std):')
_col_w = 14
_header = f'{"Measure":<35}' + ''.join(f'{c:>{_col_w}}' for c in CONDITIONS)
print(_header)
print('-' * (35 + _col_w * len(CONDITIONS)))
for name, vals_dict in panels:
    name_clean = name.replace('\n', ' ')
    row = f'{name_clean:<35}'
    for cond in CONDITIONS:
        vals = [v for v in vals_dict[cond] if np.isfinite(v)]
        if vals:
            row += f' {np.mean(vals):>{_col_w-1}.4f}±{np.std(vals):.4f}'
        else:
            row += f' {"N/A":>{_col_w}}'
    print(row)

---
## 9. Clinical Interpretation and Caveats

### What a correct result looks like

For disorders of consciousness (DoC) assessed with computational simulation:

| Measure | CNT | MCS | UWS | Mechanism |
|---------|-----|-----|-----|-----------|
| **High-SFC brain state occupancy** | Low | Medium | High | UWS brain is trapped in the SC-determined synchrony pattern; weaker long-range coupling → states determined by anatomy rather than dynamics |
| **LZc (rates)** | High | Medium | Low | Higher coupling + excitability → richer, more diverse spatiotemporal patterns |
| **PCI** | High | Medium | Low | Strong long-range coupling enables complex propagating cascades after perturbation |

### ⚠️ Important caveats

1. **LZc from spontaneous firing rates can be inverted** (UWS > CNT) when UWS is in a low-coupling, high-noise regime. Random desynchronized activity produces algorithmically complex sequences. This is why PCI is superior for DoC discrimination.

2. **BOLD brain states require long recordings** (> 10 minutes ideally). Short recordings produce noisy occupancy estimates. The Brain-Act 2-minute simulation may be borderline.

3. **PCI requires genuine perturbation simulations**. Using a virtual perturbation applied to spontaneous data (as in this notebook's synthetic demonstration) gives qualitatively correct ordering but should not be interpreted quantitatively without validation.

4. **SC-FC sorting** makes centroids comparable across subjects and studies only if the SC matrix is the same atlas and resolution. Using patient-specific SC from diffusion MRI (as in Brain-Act) is the correct approach.

5. **Brain states on firing rates vs. BOLD** answer different questions. Do not compare absolute values across domains — only within-domain comparisons are meaningful.

### Further reading

- Casali et al. (2013): Original PCI paper with TMS-EEG clinical validation
- Schartner et al. (2015): LZc and related measures in EEG and fMRI
- Demertzi et al. (2019): Four brain states from fMRI across DoC
- Deco et al. (2019): Whole-brain modelling of consciousness with the Hopf model
- Kusch et al. (2023): Brain-Act framework for computational DoC modelling

In [ ]:
# ── Save all results to CSV ───────────────────────────────────────────────────
import pandas as pd

rows = []
for cond in CONDITIONS:
    for s_idx in range(len(data[cond])):
        # LZc rate
        lzc_r = lzc_rate_vals[cond][s_idx] if s_idx < len(lzc_rate_vals[cond]) else float('nan')
        # LZc bold
        lzc_b = lzc_bold_vals[cond][s_idx] if s_idx < len(lzc_bold_vals[cond]) else float('nan')
        # PCI
        pci_v = pci_vals[cond][s_idx] if s_idx < len(pci_vals[cond]) else float('nan')
        # Brain state occupancies (rates)
        occ_rate = bs_rate[cond][s_idx].occupancy.tolist() \
            if s_idx < len(bs_rate[cond]) and bs_rate[cond][s_idx] is not None else [float('nan')]
        occ_bold_last = float('nan')
        if s_idx < len(bs_bold[cond]) and bs_bold[cond][s_idx] is not None:
            occ_bold_last = float(bs_bold[cond][s_idx].occupancy[-1])
        
        row = {
            'condition': cond,
            'subject_idx': s_idx,
            'synthetic': data[cond][s_idx].get('synthetic', False),
            'lzc_rate': lzc_r,
            'lzc_bold': lzc_b,
            'pci_mean': pci_v,
            'bs_rate_highSFC_occ': occ_rate[-1] if occ_rate else float('nan'),
            'bs_bold_highSFC_occ': occ_bold_last,
        }
        rows.append(row)

results_df = pd.DataFrame(rows)
csv_path = FIG_DIR.parent / 'metrics_v2.csv'
results_df.to_csv(csv_path, index=False)
print(f'Results saved to {csv_path}')
display(results_df.groupby('condition').agg(['mean', 'std']).round(4))

---
## Appendix A — API Quick Reference

```python
# ── Brain states ────────────────────────────────────────────────────────────
from tvbtoolkit.analysis.brain_states import summarize_brain_states, sfc_sort_centroids

# Firing rates (pipeline='firing_rate')
summary = summarize_brain_states(
    rate,                          # (time, regions) in Hz, dt=5 ms
    n_states=5,
    pipeline='firing_rate',
    bandpass_hz=(2.0, 80.0),       # 2-80 Hz for AdEx/MF
    filter_order=4,
    dt_ms=5.0,
    transient_ms=500.0,
    random_seed=42, n_init=10, max_iter=300,
)
# BOLD (pipeline='brain_act_legacy')
summary_bold = summarize_brain_states(
    bold,                          # (time, regions), TR=2.4 s
    n_states=5,
    pipeline='brain_act_legacy',
    tr_seconds=2.4,
    bandpass_hz=(0.01, 0.20),
    filter_order=3,
    random_seed=42, n_init=10, max_iter=300,
)
# Sort by SC-FC coupling
centers_s, labels_s, order, sfc = sfc_sort_centroids(summary.centers, summary.labels, sc)

# ── LZc ────────────────────────────────────────────────────────────────────
from tvbtoolkit.complexity.measures import lzc_multichannel

lzc_val = lzc_multichannel(filtered_signal)   # (time, channels), already bandpassed

# ── PCI (multi-trial, TVBSim parity) ───────────────────────────────────────
from tvbtoolkit.complexity.measures import pci_casali_like_multi_trial

mean_pci, per_trial = pci_casali_like_multi_trial(
    trials,                        # list of N_TRIALS arrays (n_regions, n_bins)
    stimulation_index=nbins_mid,   # midpoint of pre-cut window
    t_analysis_ms=300.0,           # TVBSim default
    dt_ms=5.0,
    nshuffles=10,                  # TVBSim default
    percentile=100.0,              # TVBSim default
)

# ── PCI (single-trial) ────────────────────────────────────────────────────
from tvbtoolkit.complexity.measures import pci_casali_like

pci = pci_casali_like(
    full_signal,                   # (time, regions), long continuous signal
    stimulation_index=stim_idx,    # onset in bins within full signal
    t_analysis_ms=300.0,
    dt_ms=5.0,
    nshuffles=10,
    use_post_only=True,
)
```

## Appendix B — Parameter Cheat Sheet

| Parameter | Firing-rate pipeline | BOLD pipeline | TVBSim PCI |
|-----------|---------------------|---------------|------------|
| `pipeline` | `'firing_rate'` | `'brain_act_legacy'` | N/A |
| `dt_ms` | 5.0 | N/A (TR) | 5.0 |
| `transient_ms` | 500.0 | N/A | N/A |
| Band | 2–80 Hz | 0.01–0.20 Hz | N/A |
| Filter order | 4 | 3 | N/A |
| k-means seeds | 42 | 42 | N/A |
| k-means n_init | 10 | 10 | N/A |
| `n_trials` | N/A | N/A | **5** |
| `t_analysis_ms` | N/A | N/A | **300** |
| `nshuffles` | N/A | N/A | **10** |
| `percentile` | N/A | N/A | **100.0** |